# Analyse ablation MetaSpace features

Notebook pour analyser les runs d'ablation XGBoost MetaSpace :

- classement des meilleurs sets de features,
- F1 all-to-all @ opt_train,
- precision / recall,
- F1 one-to-one,
- winrate vs `Magneto_ft_gpt`,
- effet des features TDA H0,
- analyse par fold, dataset et paire source/target.

Le notebook est robuste aux runs partiels : si le run TDA est encore en cours, il analyse ce qui est deja sauvegarde.

In [ ]:
from pathlib import Path
import json
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SNS = True
except Exception:
    HAS_SNS = False

# Retrouver la racine projet, que le notebook soit ouvert depuis notebooks/ ou depuis la racine.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'outputs').exists():
    # Fallback explicite local.
    PROJECT_ROOT = Path('/Users/nahawandkired/Documents/metamatch')

OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'exp_occidata'
FS_ROOT = OUTPUT_ROOT / 'reports' / 'meeting_baselines_vs_metamatch' / 'feature_selection_meta_space'
ANALYSIS_DIR = FS_ROOT / 'analysis_ablation_notebook'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('FS_ROOT      =', FS_ROOT)
print('ANALYSIS_DIR =', ANALYSIS_DIR)

## 1. Detection des runs disponibles

On cherche tous les dossiers contenant `combination_summary.csv`.

In [ ]:
def load_manifest(run_dir: Path):
    p = run_dir / 'run_manifest.json'
    if p.exists():
        return json.loads(p.read_text())
    return {}

run_dirs = sorted({p.parent for p in FS_ROOT.glob('*/combination_summary.csv')})
rows = []
for run_dir in run_dirs:
    manifest = load_manifest(run_dir)
    summary_path = run_dir / 'combination_summary.csv'
    try:
        n_saved = sum(1 for _ in summary_path.open('rb')) - 1
    except Exception:
        n_saved = np.nan
    rows.append({
        'run_label': run_dir.name,
        'run_dir': str(run_dir),
        'n_saved_combinations': n_saved,
        'planned_combinations': manifest.get('planned_combinations', np.nan),
        'pool_size': manifest.get('pool_size', np.nan),
        'complete': (pd.notna(n_saved) and manifest.get('planned_combinations') == n_saved),
    })

runs_df = pd.DataFrame(rows).sort_values(['complete', 'n_saved_combinations'], ascending=[False, False])
runs_df.to_csv(ANALYSIS_DIR / 'available_runs.csv', index=False)
runs_df

In [ ]:
# Choix des runs a analyser.
# Par defaut : tous les runs trouves. Tu peux restreindre cette liste si besoin.
RUN_LABELS = runs_df['run_label'].tolist()
RUN_LABELS

## 2. Chargement des summaries et enrichissement

Colonnes ajoutees :

- `has_tda` : au moins une feature TDA dans le set,
- `has_cls`, `has_syn`,
- `features_list`,
- `rank_in_run`.

In [ ]:
def split_features(s):
    if pd.isna(s) or not str(s).strip():
        return []
    return [x for x in str(s).split('|') if x]

def enrich_summary(df, run_label):
    df = df.copy()
    df['run_label'] = run_label
    df['features_list'] = df['features'].apply(split_features)
    df['has_syn'] = df['features_list'].apply(lambda xs: any(x.startswith('syn_') for x in xs))
    df['has_cls'] = df['features_list'].apply(lambda xs: any(x.startswith('cls_') for x in xs))
    df['has_tda'] = df['features_list'].apply(lambda xs: any(x.startswith('tda_') for x in xs))
    df['has_all_three'] = df['has_syn'] & df['has_cls'] & df['has_tda']
    df['feature_set_short'] = df['features'].str.replace('|', ' + ', regex=False)
    sort_cols = ['mean_f1_all2all_opt_train', 'winrate_half_ties_vs_magneto_f1_all2all', 'mean_f1_one2one']
    df = df.sort_values(sort_cols, ascending=False).reset_index(drop=True)
    df['rank_in_run'] = np.arange(1, len(df) + 1)
    return df

summaries = []
for run_label in RUN_LABELS:
    p = FS_ROOT / run_label / 'combination_summary.csv'
    if p.exists():
        summaries.append(enrich_summary(pd.read_csv(p), run_label))

summary_all = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
summary_all.to_csv(ANALYSIS_DIR / 'all_runs_combination_summary_enriched.csv', index=False)
print(summary_all.shape)
summary_all.head()

## 3. Meilleurs feature sets globaux

Classement principal :

1. `mean_f1_all2all_opt_train`,
2. `winrate_half_ties_vs_magneto_f1_all2all`,
3. `mean_f1_one2one`.

In [ ]:
TOP_COLS = [
    'run_label', 'rank_in_run', 'combo_id', 'n_features',
    'has_syn', 'has_cls', 'has_tda', 'has_all_three',
    'mean_f1_all2all_opt_train',
    'winrate_half_ties_vs_magneto_f1_all2all',
    'wins_vs_magneto_f1_all2all', 'losses_vs_magneto_f1_all2all',
    'mean_precision_all2all_opt_train', 'mean_recall_all2all_opt_train',
    'mean_f1_one2one', 'mean_delta_f1_all2all_vs_magneto',
    'features'
]

top_global = summary_all.sort_values(
    ['mean_f1_all2all_opt_train', 'winrate_half_ties_vs_magneto_f1_all2all', 'mean_f1_one2one'],
    ascending=False
)[TOP_COLS].head(50)

top_global.to_csv(ANALYSIS_DIR / 'top50_global_feature_sets.csv', index=False)
top_global

In [ ]:
# Meilleur set par run
best_by_run = summary_all.sort_values(
    ['run_label', 'mean_f1_all2all_opt_train', 'winrate_half_ties_vs_magneto_f1_all2all', 'mean_f1_one2one'],
    ascending=[True, False, False, False]
).groupby('run_label', as_index=False).head(1)[TOP_COLS]

best_by_run.to_csv(ANALYSIS_DIR / 'best_feature_set_by_run.csv', index=False)
best_by_run

## 4. Visualisations globales

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = summary_all.copy()
if HAS_SNS:
    sns.scatterplot(
        data=plot_df,
        x='n_features', y='mean_f1_all2all_opt_train',
        hue='has_tda', style='run_label', alpha=0.65, s=45
    )
else:
    for has_tda, sub in plot_df.groupby('has_tda'):
        plt.scatter(sub['n_features'], sub['mean_f1_all2all_opt_train'], label=f'has_tda={has_tda}', alpha=0.65)
plt.grid(alpha=0.25)
plt.title('F1 all-to-all @ opt_train selon nombre de features')
plt.xlabel('Nombre de features')
plt.ylabel('Mean F1 all-to-all @ opt_train')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
p = ANALYSIS_DIR / 'scatter_f1_all2all_vs_n_features.png'
plt.savefig(p, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', p)

In [ ]:
# Precision vs Recall all-to-all @ opt_train
# Couleurs forcees : bleu = configs sans TDA, vert = configs avec TDA, rouge = MetaSpace full.
plot_df = summary_all.copy()
plot_df['tda_label'] = plot_df['has_tda'].map({False: 'Reduced configs - no TDA', True: 'Reduced configs - with TDA'})

palette = {
    'Reduced configs - no TDA': '#2563eb',   # bleu fort
    'Reduced configs - with TDA': '#16a34a', # vert fort
}

fig, ax = plt.subplots(figsize=(12.5, 6.5))

# Nuage des configs reduites.
if HAS_SNS:
    sns.scatterplot(
        data=plot_df,
        x='mean_precision_all2all_opt_train',
        y='mean_recall_all2all_opt_train',
        hue='tda_label',
        palette=palette,
        alpha=0.72,
        s=58,
        edgecolor='white',
        linewidth=0.35,
        ax=ax,
    )
else:
    for label, sub in plot_df.groupby('tda_label'):
        ax.scatter(
            sub['mean_precision_all2all_opt_train'],
            sub['mean_recall_all2all_opt_train'],
            label=label,
            color=palette.get(label, '#444444'),
            alpha=0.72,
            s=58,
            edgecolor='white',
            linewidth=0.35,
        )

# Point MetaSpace full, coherent avec la figure: threshold appris sur train puis applique au test.
full_path = OUTPUT_ROOT / 'reports' / 'meeting_baselines_vs_metamatch' / 'meta_space_opt_train_threshold_by_fold.csv'
if full_path.exists():
    full_df = pd.read_csv(full_path)
    full_precision = full_df['meta_space_precision_all2all_opt_train'].mean()
    full_recall = full_df['meta_space_recall_all2all_opt_train'].mean()
    full_f1 = full_df['meta_space_f1_all2all_opt_train'].mean()
    ax.scatter(
        [full_precision], [full_recall],
        color='#dc2626',
        marker='*',
        s=420,
        edgecolor='black',
        linewidth=1.1,
        label=f'MetaSpace full @ opt_train (F1={full_f1:.3f})',
        zorder=10,
    )
    ax.annotate(
        'Full MetaSpace',
        xy=(full_precision, full_recall),
        xytext=(-105, 22),
        textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#991b1b', lw=1.2),
        fontsize=10,
        color='#991b1b',
        weight='bold',
    )
else:
    print('Full MetaSpace opt_train file missing:', full_path)

# Annoter les 3 meilleures configs reduites pour lecture rapide.
top_annot = plot_df.sort_values(
    ['mean_f1_all2all_opt_train', 'winrate_half_ties_vs_magneto_f1_all2all', 'mean_f1_one2one'],
    ascending=False,
).head(3)
for _, r in top_annot.iterrows():
    ax.annotate(
        f"#{int(r['combo_id'])} k={int(r['n_features'])}",
        xy=(r['mean_precision_all2all_opt_train'], r['mean_recall_all2all_opt_train']),
        xytext=(6, 6),
        textcoords='offset points',
        fontsize=8,
        color='#111827',
    )

ax.grid(alpha=0.25)
ax.set_title('Precision vs Recall all-to-all @ opt_train', fontsize=14, weight='bold')
ax.set_xlabel('Mean precision all-to-all @ opt_train')
ax.set_ylabel('Mean recall all-to-all @ opt_train')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, title='Configuration')
fig.tight_layout()
p = ANALYSIS_DIR / 'scatter_precision_recall_all2all_with_full.png'
fig.savefig(p, dpi=240, bbox_inches='tight')
plt.show()
print('Saved:', p)

In [ ]:
# Zoom autour du point Full MetaSpace pour voir les configs autour/en dessous.
full_path = OUTPUT_ROOT / 'reports' / 'meeting_baselines_vs_metamatch' / 'meta_space_opt_train_threshold_by_fold.csv'
if full_path.exists():
    full_df = pd.read_csv(full_path)
    full_precision = full_df['meta_space_precision_all2all_opt_train'].mean()
    full_recall = full_df['meta_space_recall_all2all_opt_train'].mean()
    full_f1 = full_df['meta_space_f1_all2all_opt_train'].mean()

    plot_df = summary_all.copy()
    plot_df['tda_label'] = plot_df['has_tda'].map({False: 'No TDA', True: 'With TDA'})
    plot_df['below_full_recall'] = plot_df['mean_recall_all2all_opt_train'] < full_recall
    plot_df['above_full_f1'] = plot_df['mean_f1_all2all_opt_train'] >= full_f1
    plot_df['near_full_precision'] = (plot_df['mean_precision_all2all_opt_train'] - full_precision).abs() <= 0.03
    plot_df['near_full_recall'] = (plot_df['mean_recall_all2all_opt_train'] - full_recall).abs() <= 0.03
    plot_df['near_full'] = plot_df['near_full_precision'] & plot_df['near_full_recall']

    relation_summary = pd.DataFrame([
        {'relation': 'configs below full recall', 'n': int(plot_df['below_full_recall'].sum())},
        {'relation': 'configs above/equal full F1', 'n': int(plot_df['above_full_f1'].sum())},
        {'relation': 'configs near full (+/-0.03 precision & recall)', 'n': int(plot_df['near_full'].sum())},
        {'relation': 'configs with precision > full precision', 'n': int((plot_df['mean_precision_all2all_opt_train'] > full_precision).sum())},
        {'relation': 'configs with recall > full recall', 'n': int((plot_df['mean_recall_all2all_opt_train'] > full_recall).sum())},
    ])
    relation_summary.to_csv(ANALYSIS_DIR / 'full_metaspace_neighborhood_counts.csv', index=False)
    display(relation_summary)

    # Fenetre de zoom : assez large pour voir les configs autour, mais pas toute la figure.
    x_min = max(0.0, full_precision - 0.22)
    x_max = min(1.02, full_precision + 0.04)
    y_min = max(0.0, full_recall - 0.22)
    y_max = min(1.02, full_recall + 0.08)
    zoom_df = plot_df[
        plot_df['mean_precision_all2all_opt_train'].between(x_min, x_max)
        & plot_df['mean_recall_all2all_opt_train'].between(y_min, y_max)
    ].copy()

    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    palette = {'No TDA': '#2563eb', 'With TDA': '#16a34a'}
    if HAS_SNS:
        sns.scatterplot(
            data=zoom_df,
            x='mean_precision_all2all_opt_train',
            y='mean_recall_all2all_opt_train',
            hue='tda_label',
            palette=palette,
            alpha=0.78,
            s=70,
            edgecolor='white',
            linewidth=0.35,
            ax=ax,
        )
    else:
        for label, sub in zoom_df.groupby('tda_label'):
            ax.scatter(sub['mean_precision_all2all_opt_train'], sub['mean_recall_all2all_opt_train'], label=label, color=palette[label], alpha=0.78, s=70)

    # Full plus petit que dans la figure globale pour ne pas masquer les points proches.
    ax.scatter(
        [full_precision], [full_recall],
        color='#dc2626', marker='*', s=220,
        edgecolor='black', linewidth=1.0,
        label=f'Full MetaSpace (F1={full_f1:.3f})',
        zorder=10,
    )
    ax.axvline(full_precision, color='#dc2626', linestyle='--', lw=1.0, alpha=0.65)
    ax.axhline(full_recall, color='#dc2626', linestyle='--', lw=1.0, alpha=0.65)
    ax.annotate('Full', xy=(full_precision, full_recall), xytext=(-34, 14), textcoords='offset points', color='#991b1b', weight='bold')

    # Annoter les meilleurs points dans le zoom.
    top_zoom = zoom_df.sort_values(['mean_f1_all2all_opt_train', 'winrate_half_ties_vs_magneto_f1_all2all'], ascending=False).head(8)
    for _, r in top_zoom.iterrows():
        ax.annotate(
            f"#{int(r['combo_id'])}",
            xy=(r['mean_precision_all2all_opt_train'], r['mean_recall_all2all_opt_train']),
            xytext=(5, 5), textcoords='offset points', fontsize=8,
        )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.grid(alpha=0.25)
    ax.set_title('Zoom Precision/Recall autour du Full MetaSpace', fontsize=13, weight='bold')
    ax.set_xlabel('Mean precision all-to-all @ opt_train')
    ax.set_ylabel('Mean recall all-to-all @ opt_train')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    fig.tight_layout()
    p = ANALYSIS_DIR / 'scatter_precision_recall_zoom_around_full.png'
    fig.savefig(p, dpi=240, bbox_inches='tight')
    plt.show()
    print('Saved:', p)
else:
    print('Missing full MetaSpace opt_train metrics:', full_path)

In [ ]:
# Top 20 en barplot
bar_df = top_global.head(20).copy()
bar_df['label'] = bar_df['run_label'].str.replace('xgb_ablation_', '', regex=False).str[:18] + ' | #' + bar_df['combo_id'].astype(str) + ' | k=' + bar_df['n_features'].astype(str)
bar_df = bar_df.iloc[::-1]
plt.figure(figsize=(12, max(6, 0.35 * len(bar_df))))
colors = ['#d1495b' if x else '#4c78a8' for x in bar_df['has_tda']]
plt.barh(bar_df['label'], bar_df['mean_f1_all2all_opt_train'], color=colors)
plt.xlabel('Mean F1 all-to-all @ opt_train')
plt.title('Top 20 feature sets, rouge = contient TDA')
plt.grid(axis='x', alpha=0.25)
plt.tight_layout()
p = ANALYSIS_DIR / 'bar_top20_feature_sets.png'
plt.savefig(p, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', p)

## 5. Effet TDA H0

Cette section compare les combinaisons avec et sans TDA, lorsque le run TDA/H0 existe deja.

In [ ]:
tda_comparison = (
    summary_all.groupby(['run_label', 'n_features', 'has_tda'], dropna=False)
    .agg(
        n_sets=('combo_id', 'count'),
        mean_f1=('mean_f1_all2all_opt_train', 'mean'),
        median_f1=('mean_f1_all2all_opt_train', 'median'),
        best_f1=('mean_f1_all2all_opt_train', 'max'),
        mean_winrate=('winrate_half_ties_vs_magneto_f1_all2all', 'mean'),
        best_winrate=('winrate_half_ties_vs_magneto_f1_all2all', 'max'),
        mean_precision=('mean_precision_all2all_opt_train', 'mean'),
        mean_recall=('mean_recall_all2all_opt_train', 'mean'),
    )
    .reset_index()
    .sort_values(['run_label', 'n_features', 'has_tda'])
)
tda_comparison.to_csv(ANALYSIS_DIR / 'tda_effect_summary_by_run_nfeatures.csv', index=False)
tda_comparison

In [ ]:
if not summary_all.empty:
    plt.figure(figsize=(12, 6))
    if HAS_SNS:
        sns.boxplot(data=summary_all, x='n_features', y='mean_f1_all2all_opt_train', hue='has_tda')
    else:
        summary_all.boxplot(column='mean_f1_all2all_opt_train', by=['n_features', 'has_tda'], rot=45)
    plt.title('Distribution F1 all-to-all selon presence de TDA')
    plt.xlabel('Nombre de features')
    plt.ylabel('Mean F1 all-to-all @ opt_train')
    plt.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    p = ANALYSIS_DIR / 'boxplot_f1_by_nfeatures_has_tda.png'
    plt.savefig(p, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', p)

In [ ]:
# Meilleurs sets avec TDA uniquement
best_with_tda = summary_all[summary_all['has_tda']].sort_values(
    ['mean_f1_all2all_opt_train', 'winrate_half_ties_vs_magneto_f1_all2all', 'mean_f1_one2one'],
    ascending=False
)[TOP_COLS].head(50)

best_with_tda.to_csv(ANALYSIS_DIR / 'top50_feature_sets_with_tda.csv', index=False)
best_with_tda

## 6. Analyse par fold pour les meilleurs sets

In [ ]:
def read_fold_metrics(run_label):
    p = FS_ROOT / run_label / 'combination_fold_metrics.csv'
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    df['run_label'] = run_label
    return df

selected = best_by_run[['run_label', 'combo_id', 'features']].copy()
# Ajouter aussi le meilleur set avec TDA si disponible.
if len(best_with_tda) > 0:
    selected = pd.concat([selected, best_with_tda[['run_label', 'combo_id', 'features']].head(3)], ignore_index=True)
selected = selected.drop_duplicates(['run_label', 'combo_id']).reset_index(drop=True)
selected

In [ ]:
fold_selected = []
for _, row in selected.iterrows():
    df = read_fold_metrics(row['run_label'])
    if df.empty:
        continue
    sub = df[df['combo_id'] == row['combo_id']].copy()
    sub['selected_features'] = row['features']
    fold_selected.append(sub)
fold_selected = pd.concat(fold_selected, ignore_index=True) if fold_selected else pd.DataFrame()
fold_selected.to_csv(ANALYSIS_DIR / 'selected_feature_sets_fold_metrics.csv', index=False)

cols = [
    'run_label', 'combo_id', 'fold_id', 'n_features',
    'f1_all2all_opt_train', 'precision_all2all_opt_train', 'recall_all2all_opt_train',
    'f1_one2one', 'magneto_f1_all2all', 'delta_f1_all2all_vs_magneto',
    'selected_features'
]
fold_selected[[c for c in cols if c in fold_selected.columns]].sort_values(['run_label','combo_id','fold_id'])

In [ ]:
if not fold_selected.empty:
    plt.figure(figsize=(12, 6))
    fold_selected['label'] = fold_selected['run_label'].str.replace('xgb_ablation_', '', regex=False).str[:20] + ' #' + fold_selected['combo_id'].astype(str)
    if HAS_SNS:
        sns.lineplot(data=fold_selected, x='fold_id', y='f1_all2all_opt_train', hue='label', marker='o')
    else:
        for label, sub in fold_selected.groupby('label'):
            plt.plot(sub['fold_id'], sub['f1_all2all_opt_train'], marker='o', label=label)
    plt.grid(alpha=0.25)
    plt.title('F1 all-to-all @ opt_train par fold')
    plt.xlabel('Fold')
    plt.ylabel('F1')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    p = ANALYSIS_DIR / 'line_selected_sets_f1_by_fold.png'
    plt.savefig(p, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', p)

## 7. Analyse par dataset et winrate dataset

Winrate dataset = sur les folds disponibles pour chaque dataset :

`(wins + 0.5 * ties) / n_folds`

avec win si `F1_MetaSpace_reduit > F1_Magneto_ft_gpt`.

In [ ]:
def read_dataset_metrics(run_label):
    p = FS_ROOT / run_label / 'combination_dataset_metrics.csv'
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    df['run_label'] = run_label
    return df

def dataset_winrate(df):
    if df.empty:
        return df
    g = df.groupby(['run_label', 'combo_id', 'features', 'dataset'], dropna=False)
    rows = []
    for key, sub in g:
        delta = sub['delta_f1_all2all_vs_magneto']
        rows.append({
            'run_label': key[0],
            'combo_id': key[1],
            'features': key[2],
            'dataset': key[3],
            'n_folds': len(sub),
            'wins': int((delta > 0).sum()),
            'ties': int((delta == 0).sum()),
            'losses': int((delta < 0).sum()),
            'winrate_half_ties': float(((delta > 0).sum() + 0.5 * (delta == 0).sum()) / len(sub)),
            'mean_delta_f1_all2all_vs_magneto': float(delta.mean()),
            'mean_f1_all2all_opt_train': float(sub['f1_all2all_opt_train'].mean()),
            'mean_magneto_f1_all2all': float(sub['magneto_f1_all2all'].mean()) if 'magneto_f1_all2all' in sub else np.nan,
            'mean_precision_all2all_opt_train': float(sub['precision_all2all_opt_train'].mean()),
            'mean_recall_all2all_opt_train': float(sub['recall_all2all_opt_train'].mean()),
            'mean_f1_one2one': float(sub['f1_one2one'].mean()),
        })
    return pd.DataFrame(rows)

dataset_selected = []
for _, row in selected.iterrows():
    df = read_dataset_metrics(row['run_label'])
    if df.empty:
        continue
    dataset_selected.append(df[df['combo_id'] == row['combo_id']].copy())
dataset_selected = pd.concat(dataset_selected, ignore_index=True) if dataset_selected else pd.DataFrame()

dataset_wr = dataset_winrate(dataset_selected)
dataset_wr.to_csv(ANALYSIS_DIR / 'selected_feature_sets_dataset_winrate.csv', index=False)
dataset_wr.sort_values(['run_label', 'combo_id', 'winrate_half_ties'], ascending=[True, True, False])

In [ ]:
if not dataset_wr.empty:
    for (run_label, combo_id), sub in dataset_wr.groupby(['run_label', 'combo_id']):
        mat = sub.pivot_table(index='dataset', values='winrate_half_ties', aggfunc='mean').sort_values('winrate_half_ties', ascending=False)
        plt.figure(figsize=(5, max(3, 0.4 * len(mat))))
        if HAS_SNS:
            sns.heatmap(mat, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1, cbar=True)
        else:
            plt.imshow(mat.values, aspect='auto', vmin=0, vmax=1)
            plt.yticks(range(len(mat.index)), mat.index)
            plt.colorbar()
        plt.title(f'Winrate dataset vs Magneto\n{run_label} | combo {combo_id}')
        plt.tight_layout()
        safe = f'{run_label}_combo{combo_id}'.replace('/', '_')
        p = ANALYSIS_DIR / f'heatmap_dataset_winrate_{safe}.png'
        plt.savefig(p, dpi=220, bbox_inches='tight')
        plt.show()
        print('Saved:', p)

## 8. Analyse par paire source/target, lecture par chunks

Le fichier pair-level peut etre tres gros. Cette cellule lit seulement les lignes du `combo_id` choisi.

In [ ]:
def load_pair_metrics_for_combo(run_label, combo_id, chunksize=300_000):
    p = FS_ROOT / run_label / 'combination_pair_metrics.csv'
    if not p.exists():
        print('Missing:', p)
        return pd.DataFrame()
    chunks = []
    for chunk in pd.read_csv(p, chunksize=chunksize):
        sub = chunk[chunk['combo_id'] == combo_id]
        if len(sub):
            chunks.append(sub.copy())
    if not chunks:
        return pd.DataFrame()
    out = pd.concat(chunks, ignore_index=True)
    out['run_label'] = run_label
    return out

# Par defaut, analyser le meilleur set global.
if len(top_global) > 0:
    PAIR_RUN_LABEL = top_global.iloc[0]['run_label']
    PAIR_COMBO_ID = int(top_global.iloc[0]['combo_id'])
    print('PAIR_RUN_LABEL =', PAIR_RUN_LABEL)
    print('PAIR_COMBO_ID   =', PAIR_COMBO_ID)
else:
    PAIR_RUN_LABEL = None
    PAIR_COMBO_ID = None

In [ ]:
if PAIR_RUN_LABEL is not None:
    pair_df = load_pair_metrics_for_combo(PAIR_RUN_LABEL, PAIR_COMBO_ID)
    pair_out = ANALYSIS_DIR / f'pair_metrics_{PAIR_RUN_LABEL}_combo{PAIR_COMBO_ID}.csv'
    pair_df.to_csv(pair_out, index=False)
    print(pair_df.shape)
    print('Saved:', pair_out)
    display(pair_df.head())

In [ ]:
if 'pair_df' in globals() and not pair_df.empty:
    pair_summary = (
        pair_df.groupby(['dataset', 'relation_type'], dropna=False)
        .agg(
            n_pairs=('pair_id', 'nunique'),
            mean_f1_all2all=('f1_all2all_opt_train', 'mean'),
            median_f1_all2all=('f1_all2all_opt_train', 'median'),
            mean_precision_all2all=('precision_all2all_opt_train', 'mean'),
            mean_recall_all2all=('recall_all2all_opt_train', 'mean'),
            mean_f1_one2one=('f1_one2one', 'mean'),
        )
        .reset_index()
        .sort_values('mean_f1_all2all', ascending=False)
    )
    pair_summary.to_csv(ANALYSIS_DIR / 'pair_summary_selected_combo_by_dataset_relation.csv', index=False)
    display(pair_summary)

## 9. Feature importance

In [ ]:
def read_importance(run_label):
    p = FS_ROOT / run_label / 'combination_feature_importance.csv'
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    df['run_label'] = run_label
    return df

importance_selected = []
for _, row in selected.iterrows():
    df = read_importance(row['run_label'])
    if df.empty:
        continue
    importance_selected.append(df[df['combo_id'] == row['combo_id']].copy())
importance_selected = pd.concat(importance_selected, ignore_index=True) if importance_selected else pd.DataFrame()

if not importance_selected.empty:
    imp_summary = (
        importance_selected.groupby(['run_label', 'combo_id', 'feature', 'family'], dropna=False)['xgb_importance']
        .agg(mean_importance='mean', std_importance='std', min_importance='min', max_importance='max')
        .reset_index()
        .sort_values(['run_label', 'combo_id', 'mean_importance'], ascending=[True, True, False])
    )
else:
    imp_summary = pd.DataFrame()

imp_summary.to_csv(ANALYSIS_DIR / 'selected_feature_sets_importance_summary.csv', index=False)
imp_summary

In [ ]:
if not imp_summary.empty:
    for (run_label, combo_id), sub in imp_summary.groupby(['run_label', 'combo_id']):
        top = sub.sort_values('mean_importance', ascending=False).iloc[::-1]
        plt.figure(figsize=(8, max(3, 0.35 * len(top))))
        colors = top['family'].map({'syn':'#4c78a8', 'cls':'#f58518', 'tda':'#d1495b'}).fillna('#999999')
        plt.barh(top['feature'], top['mean_importance'], color=colors)
        plt.xlabel('Mean XGBoost importance')
        plt.title(f'Feature importance | {run_label} | combo {combo_id}')
        plt.grid(axis='x', alpha=0.25)
        plt.tight_layout()
        safe = f'{run_label}_combo{combo_id}'.replace('/', '_')
        p = ANALYSIS_DIR / f'importance_{safe}.png'
        plt.savefig(p, dpi=220, bbox_inches='tight')
        plt.show()
        print('Saved:', p)

## 10. Exports finaux

Tous les tableaux/figures produits par ce notebook sont dans `analysis_ablation_notebook`.

In [1]:
from pathlib import Path
import pandas as pd

# =========================
# Paths
# =========================
downloads = Path.home() / "Downloads"

baseline_files = [
    downloads / "baseline.csv",
    downloads / "baseline2.csv",
]

out_full = downloads / "combined_baselines_no_metaspace.csv"
out_summary = downloads / "combined_baselines_summary_no_metaspace.csv"

# =========================
# Load
# =========================
dfs = []

for f in baseline_files:
    if not f.exists():
        print(f"WARNING: file not found: {f}")
        continue

    df = pd.read_csv(f, low_memory=False)
    df["source_file"] = f.name
    dfs.append(df)

if not dfs:
    raise FileNotFoundError("Aucun fichier baseline trouvé dans Downloads.")

df = pd.concat(dfs, ignore_index=True)

# =========================
# Clean columns
# =========================
for col in ["F1", "All_Precision", "All_Recall", "Recall@GT", "runtime"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove Meta Space
df = df[df["method"].astype(str).str.lower().str.strip() != "meta space"].copy()

# =========================
# Remove exact duplicate rows
# =========================
df = df.drop_duplicates()

# =========================
# Keep best duplicated Magneto rows
# =========================
# Key = same experiment/pair/method.
# If the same Magneto result exists in both files, keep the row with best F1.
key_cols = [
    "scenario",
    "method",
    "dataset",
    "source_table",
    "target_table",
    "model",
    "benchmark",
    "type",
]

key_cols = [c for c in key_cols if c in df.columns]

is_magneto = df["method"].astype(str).str.lower().str.contains("magneto", na=False)

df_non_magneto = df[~is_magneto].copy()
df_magneto = df[is_magneto].copy()

df_magneto = (
    df_magneto
    .sort_values("F1", ascending=False)
    .drop_duplicates(subset=key_cols, keep="first")
)

df_final = pd.concat([df_non_magneto, df_magneto], ignore_index=True)

# Remove duplicates again after merge
df_final = df_final.drop_duplicates()

# =========================
# Summary without one-to-one
# =========================
summary = (
    df_final
    .groupby("method", dropna=False)
    .agg(
        n_rows=("F1", "size"),
        n_ok_f1=("F1", "count"),
        f1_all_to_all_mean=("F1", "mean"),
        f1_all_to_all_std=("F1", "std"),
        precision_all_to_all_mean=("All_Precision", "mean"),
        recall_all_to_all_mean=("All_Recall", "mean"),
        f1_at_ground_size_mean=("Recall@GT", "mean"),
        f1_at_ground_size_std=("Recall@GT", "std"),
        runtime_mean=("runtime", "mean"),
    )
    .reset_index()
    .sort_values("f1_all_to_all_mean", ascending=False)
)

# =========================
# Save
# =========================
df_final.to_csv(out_full, index=False)
summary.to_csv(out_summary, index=False)

print("Saved full combined file:")
print(out_full)

print("\nSaved summary file:")
print(out_summary)

print("\nRanking by F1 all-to-all:")
print(
    summary[
        [
            "method",
            "f1_all_to_all_mean",
            "f1_at_ground_size_mean",
            "runtime_mean",
            "n_rows",
        ]
    ].to_string(index=False)
)

Saved full combined file:
/Users/nahawandkired/Downloads/combined_baselines_no_metaspace.csv

Saved summary file:
/Users/nahawandkired/Downloads/combined_baselines_summary_no_metaspace.csv

Ranking by F1 all-to-all:
      method  f1_all_to_all_mean  f1_at_ground_size_mean  runtime_mean  n_rows
MagnetoFTGPT            0.719325                0.730220     92.950017     578
  MagnetoGPT            0.704167                0.667410    104.227049    1884
        Coma            0.568107                0.601641      1.685612    3486
 SimFlooding            0.564425                0.531489      3.209708    3786
      Coma++            0.552333                0.691808     99.934043    4096
     Magneto            0.508945                0.576971      7.934021    9865
   MagnetoFT            0.484200                0.566759      7.664805    3204
    ISResMat            0.457514                0.407102    111.232971    3302
       Cupid            0.346236                0.325217     59.216399   

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOTS = [
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_valentine_3runs_local"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_valentine_3runs_local/run_01"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_repeats_zero_baselines_fixed_01"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_repeats_local_fixed_01"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_repeats"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_missing_baselines_fixed"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_baselines_5runs_all"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata"),
    Path("/Users/nahawandkired/Downloads"),
]

OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CANDIDATES = OUT_DIR / "candidate_files.csv"
OUT_LONG = OUT_DIR / "all_baseline_results_long.csv"
OUT_SUMMARY = OUT_DIR / "all_baselines_best_summary.csv"
OUT_COVERAGE = OUT_DIR / "coverage_by_method_source.csv"

METHOD_MAP = {
    "Coma": "COMA",
    "coma": "COMA",
    "coma_schema": "COMA Schema",
    "coma_instance": "COMA Instance",
    "coma_inst": "COMA Instance",
    "Coma++": "COMA++",
    "coma_pp": "COMA++",
    "Cupid": "Cupid",
    "cupid": "Cupid",
    "cupid_ext": "Cupid Ext",
    "SimFlooding": "Similarity Flooding",
    "similarity_flooding": "Similarity Flooding",
    "similarity_flooding_ext": "Similarity Flooding Ext",
    "Distribution": "Distribution Based",
    "distribution_based": "Distribution Based",
    "distribution_based_ext": "Distribution Based Ext",
    "ISResMat": "ISResMat",
    "LLMATCH": "LLMATCH",
    "Magneto": "Magneto",
    "MagnetoFT": "MagnetoFT",
    "MagnetoGPT": "MagnetoGPT",
    "MagnetoFTGPT": "MagnetoFTGPT",
    "Magneto_no_ft_no_gpt": "Magneto",
    "Magneto_ft_no_gpt": "MagnetoFT",
    "Magneto_no_ft_gpt": "MagnetoGPT",
    "Magneto_ft_gpt": "MagnetoFTGPT",
    "SMUTF": "SMUTF",
    "smutf": "SMUTF",
}

KNOWN = set(METHOD_MAP.values())

def normalize_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def pick_col(cols, candidates):
    exact = {str(c).lower(): c for c in cols}
    norm = {normalize_key(c): c for c in cols}
    for cand in candidates:
        if str(cand).lower() in exact:
            return exact[str(cand).lower()]
    for cand in candidates:
        if normalize_key(cand) in norm:
            return norm[normalize_key(cand)]
    return None

def normalize_method(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return METHOD_MAP.get(x, x)

def make_pair_id(df):
    cols = set(df.columns)

    if "pair_id" in cols:
        return df["pair_id"].astype(str)

    if {"dataset", "source_table", "target_table"}.issubset(cols):
        return df["dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)

    if {"id_dataset", "source_table", "target_table"}.issubset(cols):
        return df["id_dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)

    if {"dataset", "source", "target"}.issubset(cols):
        return df["dataset"].astype(str) + "__" + df["source"].astype(str) + "__" + df["target"].astype(str)

    if {"source", "target"}.issubset(cols):
        return df["source"].astype(str) + "__" + df["target"].astype(str)

    return pd.Series([np.nan] * len(df), index=df.index)

F1_ALL_COLS = [
    "F1",
    "All_F1Score",
    "All_F1",
    "all_f1score",
    "f1_all2all_mean",
    "f1_all_to_all_mean",
    "metric_f1_at_05",
    "f1_all2all_opt_train",
    "mean_f1_all2all_opt_train",
    "mean_f1_pairs",
    "mean_f1_551_pair_ids",
    "mean_f1_folds",
    "mean_f1_fold",
    "mean_f1",
    "f1",
]

PREC_ALL_COLS = [
    "All_Precision",
    "metric_precision_at_05",
    "precision_all2all_opt_train",
    "mean_precision_pairs",
    "mean_precision_551_pair_ids",
    "mean_precision_folds",
    "mean_precision_fold",
    "precision",
]

REC_ALL_COLS = [
    "All_Recall",
    "metric_recall_at_05",
    "recall_all2all_opt_train",
    "mean_recall_pairs",
    "mean_recall_551_pair_ids",
    "mean_recall_folds",
    "mean_recall_fold",
    "recall",
]

F1_GT_COLS = [
    "Recall@GT",
    "All_RecallAtSizeofGroundTruth",
    "F1@GT",
    "f1_ground_size",
    "mean_f1_ground_size",
    "f1_at_ground_size",
    "f1_at_ground_size_mean",
    "recall_ground_size",
    "mean_recall_ground_size",
]

RUNTIME_COLS = ["runtime", "runtime_mean", "runtime_sec", "elapsed", "time"]

candidate_files = []
all_rows = []

for root in ROOTS:
    if not root.exists():
        print("Missing root:", root)
        continue

    for f in root.rglob("*.csv"):
        if "1to1" in f.name.lower() or "one2one" in f.name.lower():
            continue

        try:
            head = pd.read_csv(f, nrows=5, low_memory=False)
        except Exception:
            continue

        cols = list(head.columns)

        method_col = pick_col(cols, ["method", "classifier", "method_norm", "matcher", "algorithm"])
        f1_col = pick_col(cols, F1_ALL_COLS)
        gt_col = pick_col(cols, F1_GT_COLS)

        if method_col is None and f1_col is None and gt_col is None:
            continue

        candidate_files.append({
            "file": str(f),
            "method_col": method_col,
            "f1_col": f1_col,
            "gt_col": gt_col,
            "columns": " | ".join(cols),
        })

        try:
            df = pd.read_csv(f, low_memory=False)
        except Exception:
            continue

        cols = list(df.columns)

        method_col = pick_col(cols, ["method", "classifier", "method_norm", "matcher", "algorithm"])
        if method_col is None:
            continue

        f1_col = pick_col(cols, F1_ALL_COLS)
        precision_col = pick_col(cols, PREC_ALL_COLS)
        recall_col = pick_col(cols, REC_ALL_COLS)
        gt_col = pick_col(cols, F1_GT_COLS)
        runtime_col = pick_col(cols, RUNTIME_COLS)

        out = pd.DataFrame()
        out["method"] = df[method_col].map(normalize_method)
        out["pair_id"] = make_pair_id(df)
        out["source_file"] = str(f)

        out["f1_all_to_all"] = pd.to_numeric(df[f1_col], errors="coerce") if f1_col else np.nan
        out["precision_all_to_all"] = pd.to_numeric(df[precision_col], errors="coerce") if precision_col else np.nan
        out["recall_all_to_all"] = pd.to_numeric(df[recall_col], errors="coerce") if recall_col else np.nan
        out["f1_at_ground_size"] = pd.to_numeric(df[gt_col], errors="coerce") if gt_col else np.nan
        out["runtime"] = pd.to_numeric(df[runtime_col], errors="coerce") if runtime_col else np.nan

        out["used_f1_col"] = f1_col
        out["used_precision_col"] = precision_col
        out["used_recall_col"] = recall_col
        out["used_gt_col"] = gt_col

        out = out[~out["method"].astype(str).str.lower().str.contains("meta_space|meta space|metaspace", na=False)]
        out = out[out["method"].isin(KNOWN)]
        out = out[out[["f1_all_to_all", "f1_at_ground_size"]].notna().any(axis=1)]

        if len(out):
            all_rows.append(out)

candidates = pd.DataFrame(candidate_files)
candidates.to_csv(OUT_CANDIDATES, index=False)

long = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
long.to_csv(OUT_LONG, index=False)

coverage = (
    long.groupby(["method", "source_file"])
    .agg(
        n_rows=("method", "size"),
        n_pairs=("pair_id", lambda x: x.dropna().nunique()),
        f1_all_mean=("f1_all_to_all", "mean"),
        f1_gt_mean=("f1_at_ground_size", "mean"),
        used_f1_col=("used_f1_col", lambda x: sorted(set(x.dropna().astype(str)))),
        used_gt_col=("used_gt_col", lambda x: sorted(set(x.dropna().astype(str)))),
    )
    .reset_index()
)

coverage["dist_551"] = (coverage["n_pairs"] - 551).abs()
coverage = coverage.sort_values(
    ["method", "dist_551", "n_pairs", "f1_all_mean"],
    ascending=[True, True, False, False]
)
coverage.to_csv(OUT_COVERAGE, index=False)

final_rows = []

for method, g in long.groupby("method"):
    row = {"method": method}

    for metric, prefix in [
        ("f1_all_to_all", "all_to_all"),
        ("f1_at_ground_size", "ground_size"),
    ]:
        gg = g[g[metric].notna()].copy()
        if gg.empty:
            row[f"f1_{prefix}_mean"] = np.nan
            row[f"f1_{prefix}_std"] = np.nan
            row[f"n_pairs_{prefix}"] = 0
            row[f"source_{prefix}"] = ""
            continue

        by_source = (
            gg.groupby("source_file")
            .agg(
                n_pairs=("pair_id", lambda x: x.dropna().nunique()),
                mean=(metric, "mean"),
                std=(metric, "std"),
            )
            .reset_index()
        )

        by_source["dist_551"] = (by_source["n_pairs"] - 551).abs()
        by_source = by_source.sort_values(
            ["dist_551", "n_pairs", "mean"],
            ascending=[True, False, False]
        )

        best_source = by_source.iloc[0]["source_file"]
        part = gg[gg["source_file"] == best_source].copy()

        if part["pair_id"].notna().any():
            part = (
                part.sort_values(metric, ascending=False)
                .drop_duplicates(subset=["method", "pair_id"], keep="first")
            )

        row[f"f1_{prefix}_mean"] = part[metric].mean()
        row[f"f1_{prefix}_std"] = part[metric].std()
        row[f"n_pairs_{prefix}"] = part["pair_id"].dropna().nunique()
        row[f"source_{prefix}"] = best_source

        if prefix == "all_to_all":
            row["precision_all_to_all_mean"] = part["precision_all_to_all"].mean()
            row["recall_all_to_all_mean"] = part["recall_all_to_all"].mean()
            row["runtime_mean"] = part["runtime"].mean()
            row["used_f1_col"] = part["used_f1_col"].dropna().iloc[0] if part["used_f1_col"].notna().any() else None

    final_rows.append(row)

summary = pd.DataFrame(final_rows)
summary = summary.sort_values("f1_all_to_all_mean", ascending=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("Candidate files:", len(candidates))
print("Loaded result rows:", len(long))

print("\nSaved:")
print(OUT_CANDIDATES)
print(OUT_LONG)
print(OUT_COVERAGE)
print(OUT_SUMMARY)

print("\nBest baseline summary:")
print(summary.to_string(index=False))

Candidate files: 352
Loaded result rows: 342405

Saved:
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/candidate_files.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/all_baseline_results_long.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/coverage_by_method_source.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/all_baselines_best_summary.csv

Best baseline summary:
                 method  f1_all_to_all_mean  f1_all_to_all_std  n_pairs_all_to_all                                                                                                                                                                             source_all_to_all  precision_all_to_all_mean  recall_all_to_all_mean  runtime_mean     used_f1_col  f1_ground_size_mean  f1_ground_size_std  n_pairs_ground_size                                                                                                  

In [1]:

from pathlib import Path
import pandas as pd

ROOTS = [
    Path("/Users/nahawandkired/Documents/metamatch/outputs"),
    Path.home() / "Downloads",
    Path("/Users/nahawandkired/Documents/Recherche/baselines/results")

]

rows = []

for root in ROOTS:
    if not root.exists():
        continue

    for f in root.rglob("*.csv"):
        try:
            df = pd.read_csv(f, nrows=0, low_memory=False)
        except Exception as e:
            rows.append({
                "file": str(f),
                "n_cols": -1,
                "columns": f"ERROR: {e}"
            })
            continue

        rows.append({
            "file": str(f),
            "n_cols": len(df.columns),
            "columns": " ||| ".join(df.columns)
        })

res = pd.DataFrame(rows)
res = res.sort_values("file")

out = "/Users/nahawandkired/Documents/metamatch/outputs/all_csv_columns.csv"
res.to_csv(out, index=False)

print(f"Saved: {out}")
print(f"Nb CSV: {len(res)}")

print("\n===== FIRST 50 FILES =====\n")
for _, r in res.head(50).iterrows():
    print("\nFILE:")
    print(r["file"])
    print("NCOLS:", r["n_cols"])
    print("COLUMNS:")
    print(r["columns"])
    print("-"*120)


Saved: /Users/nahawandkired/Documents/metamatch/outputs/all_csv_columns.csv
Nb CSV: 1198

===== FIRST 50 FILES =====


FILE:
/Users/nahawandkired/Documents/Recherche/baselines/results/baseline.csv
NCOLS: 37
COLUMNS:
scenario ||| method ||| All_Precision ||| F1 ||| All_Recall ||| Recall@GT ||| dataset ||| Category ||| model ||| merged_source_file ||| source_result_file ||| benchmark ||| type ||| source_table ||| target_table ||| ncols_src ||| ncols_tgt ||| nrows_src ||| nrows_tgt ||| nmatches ||| runtime ||| MRR ||| All_PrecisionTop10Percent ||| One2One_Precision ||| One2One_F1Score ||| One2One_Recall ||| One2One_PrecisionTop10Percent ||| One2One_RecallAtSizeofGroundTruth ||| model_base ||| embedding_model ||| classifier ||| repeat ||| seed ||| nb_test ||| nb_0 ||| nb_1 ||| scenario_label
------------------------------------------------------------------------------------------------------------------------

FILE:
/Users/nahawandkired/Documents/Recherche/baselines/results/baseline2.csv


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

ROOTS = [
    Path("/Users/nahawandkired/Documents/Recherche/baselines/results"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs"),
    Path("/Users/nahawandkired/Downloads"),
]
OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CANDIDATES = OUT_DIR / "candidate_files.csv"
OUT_LONG = OUT_DIR / "all_baseline_results_long.csv"
OUT_SUMMARY = OUT_DIR / "all_baselines_best_summary.csv"
OUT_COVERAGE = OUT_DIR / "coverage_by_method_source.csv"

METHOD_MAP = {
    "Coma": "COMA", "coma": "COMA",
    "coma_schema": "COMA Schema",
    "coma_instance": "COMA Instance",
    "coma_inst": "COMA Instance",
    "Coma++": "COMA++", "coma_pp": "COMA++",
    "Cupid": "Cupid", "cupid": "Cupid", "cupid_ext": "Cupid Ext",
    "SimFlooding": "Similarity Flooding",
    "similarity_flooding": "Similarity Flooding",
    "similarity_flooding_ext": "Similarity Flooding Ext",
    "Distribution": "Distribution Based",
    "distribution_based": "Distribution Based",
    "distribution_based_ext": "Distribution Based Ext",
    "ISResMat": "ISResMat",
    "LLMATCH": "LLMATCH", "LLMatch": "LLMATCH", "llmatch": "LLMATCH",
    "Magneto": "Magneto",
    "MagnetoFT": "MagnetoFT",
    "MagnetoGPT": "MagnetoGPT",
    "MagnetoFTGPT": "MagnetoFTGPT",
    "Magneto_no_ft_no_gpt": "Magneto",
    "Magneto_ft_no_gpt": "MagnetoFT",
    "Magneto_no_ft_gpt": "MagnetoGPT",
    "Magneto_ft_gpt": "MagnetoFTGPT",
    "SMUTF": "SMUTF", "smutf": "SMUTF",
}

KNOWN = set(METHOD_MAP.values())

def normalize_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def pick_col(cols, candidates):
    exact = {str(c).lower(): c for c in cols}
    norm = {normalize_key(c): c for c in cols}

    for cand in candidates:
        if str(cand).lower() in exact:
            return exact[str(cand).lower()]

    for cand in candidates:
        k = normalize_key(cand)
        if k in norm:
            return norm[k]

    return None

def normalize_method(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return METHOD_MAP.get(x, x)

def method_from_path(path):
    p = str(path).lower()
    if "llmatch" in p:
        return "LLMATCH"
    if "smutf" in p:
        return "SMUTF"
    if "magnetoftgpt" in p or "magneto_ft_gpt" in p:
        return "MagnetoFTGPT"
    if "magnetogpt" in p or "magneto_no_ft_gpt" in p:
        return "MagnetoGPT"
    if "magnetoft" in p or "magneto_ft_no_gpt" in p:
        return "MagnetoFT"
    if "magneto" in p:
        return "Magneto"
    return np.nan

def make_pair_id(df):
    cols = set(df.columns)

    if "pair_id" in cols:
        return df["pair_id"].astype(str)

    if {"dataset", "source_table", "target_table"}.issubset(cols):
        return df["dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)

    if {"benchmark", "dataset", "source_table", "target_table"}.issubset(cols):
        return df["benchmark"].astype(str) + "__" + df["dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)

    if {"id_dataset", "source_table", "target_table"}.issubset(cols):
        return df["id_dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)

    if {"dataset", "source", "target"}.issubset(cols):
        return df["dataset"].astype(str) + "__" + df["source"].astype(str) + "__" + df["target"].astype(str)

    if {"source", "target"}.issubset(cols):
        return df["source"].astype(str) + "__" + df["target"].astype(str)

    return pd.Series([np.nan] * len(df), index=df.index)

F1_ALL_COLS = [
    "F1", "All_F1Score", "All_F1", "all_f1score",
    "f1_all2all_mean", "f1_all_to_all_mean",
    "metric_f1_at_05",
    "f1_all2all_opt_train", "mean_f1_all2all_opt_train",
    "mean_f1_pairs", "mean_f1_551_pair_ids",
    "mean_f1_folds", "mean_f1_fold", "mean_f1",
    "f1", "f1_score",
]

PREC_ALL_COLS = [
    "All_Precision", "metric_precision_at_05",
    "precision_all2all_opt_train",
    "mean_precision_pairs", "mean_precision_551_pair_ids",
    "mean_precision_folds", "mean_precision_fold",
    "precision", "Precision",
]

REC_ALL_COLS = [
    "All_Recall", "metric_recall_at_05",
    "recall_all2all_opt_train",
    "mean_recall_pairs", "mean_recall_551_pair_ids",
    "mean_recall_folds", "mean_recall_fold",
    "recall", "Recall",
]

F1_GT_COLS = [
    "All_F1AtSizeofGroundTruth",
    "Recall@GT",
    "All_RecallAtSizeofGroundTruth",
    "F1@GT",
    "f1_ground_size",
    "mean_f1_ground_size",
    "f1_at_ground_size",
    "f1_at_ground_size_mean",
    "recall_ground_size",
    "mean_recall_ground_size",
]

F1_1TO1_COLS = [
    "One2One_F1Score",
    "One2One_F1AtSizeofGroundTruth",
    "One2One_F1",
    "f1_1to1",
    "metric_f1_1to1",
    "mean_f1_1to1",
    "mean_f1_one2one",
    "mean_f1_one2one_pairs",
]
RUNTIME_COLS = ["runtime", "runtime_mean", "runtime_sec", "elapsed", "time"]

def load_any_file(path):
    suffix = path.suffix.lower()

    try:
        if suffix == ".csv":
            return pd.read_csv(path, low_memory=False)
        if suffix == ".json":
            with open(path, "r") as f:
                data = json.load(f)
            if isinstance(data, dict):
                return pd.DataFrame([data])
            return pd.DataFrame(data)
        if suffix == ".jsonl":
            return pd.read_json(path, lines=True)
        if suffix == ".parquet":
            return pd.read_parquet(path)
        if suffix in [".xlsx", ".xls"]:
            return pd.read_excel(path)
    except Exception:
        return None

    return None

candidate_files = []
all_rows = []

for root in ROOTS:
    if not root.exists():
        print("Missing root:", root)
        continue

    for f in root.rglob("*"):
        if not f.is_file():
            continue

        if f.suffix.lower() not in [".csv", ".json", ".jsonl", ".parquet", ".xlsx", ".xls"]:
            continue

        # on garde one-to-one comme métrique séparée, donc on n'exclut plus les fichiers 1to1

        df = load_any_file(f)
        if df is None or df.empty:
            continue

        cols = list(df.columns)

        method_col = pick_col(cols, ["method", "classifier", "method_norm", "matcher", "algorithm"])
        f1_col = pick_col(cols, F1_ALL_COLS)
        precision_col = pick_col(cols, PREC_ALL_COLS)
        recall_col = pick_col(cols, REC_ALL_COLS)
        gt_col = pick_col(cols, F1_GT_COLS)
        one_col = pick_col(cols, F1_1TO1_COLS)
        runtime_col = pick_col(cols, RUNTIME_COLS)

        if method_col is None and pd.isna(method_from_path(f)):
            continue

        if f1_col is None and gt_col is None and one_col is None and precision_col is None and recall_col is None:
            continue

        candidate_files.append({
            "file": str(f),
            "method_col": method_col,
            "f1_col": f1_col,
            "gt_col": gt_col,
            "one_to_one_col": one_col,
            "precision_col": precision_col,
            "recall_col": recall_col,
            "columns": " | ".join(cols),
        })

        out = pd.DataFrame()

        if method_col is not None:
            out["method"] = df[method_col].map(normalize_method)
        else:
            out["method"] = method_from_path(f)

        out["pair_id"] = make_pair_id(df)
        out["source_file"] = str(f)

        out["f1_all_to_all"] = pd.to_numeric(df[f1_col], errors="coerce") if f1_col else np.nan
        out["precision_all_to_all"] = pd.to_numeric(df[precision_col], errors="coerce") if precision_col else np.nan
        out["recall_all_to_all"] = pd.to_numeric(df[recall_col], errors="coerce") if recall_col else np.nan
        out["f1_ground_size"] = pd.to_numeric(df[gt_col], errors="coerce") if gt_col else np.nan
        out["f1_one_to_one"] = pd.to_numeric(df[one_col], errors="coerce") if one_col else np.nan
        out["runtime"] = pd.to_numeric(df[runtime_col], errors="coerce") if runtime_col else np.nan

        out["used_f1_col"] = f1_col
        out["used_precision_col"] = precision_col
        out["used_recall_col"] = recall_col
        out["used_ground_size_col"] = gt_col
        out["used_one_to_one_col"] = one_col

        out = out[~out["method"].astype(str).str.lower().str.contains("meta_space|meta space|metaspace|metamatch", na=False)]
        out = out[out["method"].isin(KNOWN)]

        metric_cols = ["f1_all_to_all", "f1_ground_size", "f1_one_to_one", "precision_all_to_all", "recall_all_to_all"]
        out = out[out[metric_cols].notna().any(axis=1)]

        if len(out):
            all_rows.append(out)

candidates = pd.DataFrame(candidate_files)
candidates.to_csv(OUT_CANDIDATES, index=False)

long = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
long.to_csv(OUT_LONG, index=False)

print("Candidate files:", len(candidates))
print("Loaded result rows:", len(long))

coverage = (
    long.groupby(["method", "source_file"])
    .agg(
        n_rows=("method", "size"),
        n_pairs=("pair_id", lambda x: x.dropna().nunique()),
        f1_all_mean=("f1_all_to_all", "mean"),
        f1_gt_mean=("f1_ground_size", "mean"),
        f1_1to1_mean=("f1_one_to_one", "mean"),
        precision_mean=("precision_all_to_all", "mean"),
        recall_mean=("recall_all_to_all", "mean"),
        runtime_mean=("runtime", "mean"),
        used_f1_col=("used_f1_col", lambda x: sorted(set(x.dropna().astype(str)))),
        used_gt_col=("used_ground_size_col", lambda x: sorted(set(x.dropna().astype(str)))),
        used_1to1_col=("used_one_to_one_col", lambda x: sorted(set(x.dropna().astype(str)))),
    )
    .reset_index()
)

coverage["dist_551"] = (coverage["n_pairs"] - 551).abs()

coverage = coverage.sort_values(
    ["method", "dist_551", "n_pairs", "f1_all_mean", "f1_gt_mean"],
    ascending=[True, True, False, False, False]
)

coverage.to_csv(OUT_COVERAGE, index=False)

final_rows = []

for method, g in long.groupby("method"):
    row = {"method": method}

    for metric, prefix in [
        ("f1_all_to_all", "all_to_all"),
        ("f1_ground_size", "ground_size"),
        ("f1_one_to_one", "one_to_one"),
    ]:
        gg = g[g[metric].notna()].copy()

        if gg.empty:
            row[f"f1_{prefix}_mean"] = np.nan
            row[f"f1_{prefix}_std"] = np.nan
            row[f"n_pairs_{prefix}"] = 0
            row[f"source_{prefix}"] = ""
            continue

        by_source = (
            gg.groupby("source_file")
            .agg(
                n_rows=("method", "size"),
                n_pairs=("pair_id", lambda x: x.dropna().nunique()),
                mean=(metric, "mean"),
                std=(metric, "std"),
            )
            .reset_index()
        )

        by_source["dist_551"] = (by_source["n_pairs"] - 551).abs()

        # préférence :
        # 1. proche de 551 paires
        # 2. plus grand nombre de paires
        # 3. meilleur F1
        by_source = by_source.sort_values(
            ["dist_551", "n_pairs", "mean"],
            ascending=[True, False, False]
        )

        best_source = by_source.iloc[0]["source_file"]
        part = gg[gg["source_file"] == best_source].copy()

        if part["pair_id"].notna().any():
            part = (
                part.sort_values(metric, ascending=False)
                .drop_duplicates(subset=["method", "pair_id"], keep="first")
            )

        row[f"f1_{prefix}_mean"] = part[metric].mean()
        row[f"f1_{prefix}_std"] = part[metric].std()
        row[f"n_pairs_{prefix}"] = part["pair_id"].dropna().nunique()
        row[f"source_{prefix}"] = best_source

        if prefix == "all_to_all":
            row["precision_all_to_all_mean"] = part["precision_all_to_all"].mean()
            row["recall_all_to_all_mean"] = part["recall_all_to_all"].mean()
            row["runtime_mean"] = part["runtime"].mean()
            row["used_f1_col"] = part["used_f1_col"].dropna().iloc[0] if part["used_f1_col"].notna().any() else None

    final_rows.append(row)

summary = pd.DataFrame(final_rows)
summary = summary.sort_values("f1_all_to_all_mean", ascending=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("\nSaved:")
print(OUT_CANDIDATES)
print(OUT_LONG)
print(OUT_COVERAGE)
print(OUT_SUMMARY)

print("\nBest baseline summary:")
print(summary.to_string(index=False))

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

# ============================================================
# CONFIG
# ============================================================

ROOTS = [
    Path("/Users/nahawandkired/Documents/Recherche/baselines/results"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs"),
    Path("/Users/nahawandkired/Downloads"),
]

OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CANDIDATES = OUT_DIR / "candidate_files.csv"
OUT_LONG = OUT_DIR / "all_baseline_results_long.csv"
OUT_COVERAGE = OUT_DIR / "coverage_by_method_source.csv"
OUT_SUMMARY = OUT_DIR / "all_baselines_best_summary.csv"

# ============================================================
# METHODS
# ============================================================

METHOD_MAP = {
    "Coma": "COMA",
    "coma": "COMA",
    "coma_schema": "COMA Schema",
    "coma_instance": "COMA Instance",
    "coma_inst": "COMA Instance",

    "Coma++": "COMA++",
    "coma_pp": "COMA++",

    "Cupid": "Cupid",
    "cupid": "Cupid",
    "cupid_ext": "Cupid Ext",

    "SimFlooding": "Similarity Flooding",
    "similarity_flooding": "Similarity Flooding",
    "similarity_flooding_ext": "Similarity Flooding Ext",

    "Distribution": "Distribution Based",
    "distribution_based": "Distribution Based",
    "distribution_based_ext": "Distribution Based Ext",

    "ISResMat": "ISResMat",

    "LLMATCH": "LLMATCH",
    "LLMatch": "LLMATCH",
    "llmatch": "LLMATCH",

    "Magneto": "Magneto",
    "MagnetoFT": "MagnetoFT",
    "MagnetoGPT": "MagnetoGPT",
    "MagnetoFTGPT": "MagnetoFTGPT",

    "Magneto_no_ft_no_gpt": "Magneto",
    "Magneto_ft_no_gpt": "MagnetoFT",
    "Magneto_no_ft_gpt": "MagnetoGPT",
    "Magneto_ft_gpt": "MagnetoFTGPT",

    "SMUTF": "SMUTF",
    "smutf": "SMUTF",
}

KNOWN = set(METHOD_MAP.values())

# ============================================================
# COLUMN CANDIDATES
# ============================================================

F1_ALL_COLS = [
    "F1",
    "All_F1Score",
    "All_F1",
    "all_f1score",
    "f1",
    "f1_score",
    "f1_all2all",
    "f1_all2all_mean",
    "f1_all_to_all_mean",
    "metric_f1_at_05",
    "f1_all2all_opt_train",
    "mean_f1_all2all_opt_train",
    "mean_f1",
    "mean_f1_fold",
    "mean_f1_folds",
    "mean_f1_pairs",
    "mean_f1_pairfold",
    "mean_f1_551_pair_ids",
]

PREC_ALL_COLS = [
    "All_Precision",
    "precision",
    "Precision",
    "metric_precision_at_05",
    "precision_all2all_opt_train",
    "mean_precision",
    "mean_precision_fold",
    "mean_precision_folds",
    "mean_precision_pairs",
    "mean_precision_pairfold",
    "mean_precision_551_pair_ids",
]

REC_ALL_COLS = [
    "All_Recall",
    "recall",
    "Recall",
    "metric_recall_at_05",
    "recall_all2all_opt_train",
    "mean_recall",
    "mean_recall_fold",
    "mean_recall_folds",
    "mean_recall_pairs",
    "mean_recall_pairfold",
    "mean_recall_551_pair_ids",
]

F1_GT_COLS = [
    "All_F1AtSizeofGroundTruth",
    "Recall@GT",
    "F1@GT",
    "All_RecallAtSizeofGroundTruth",
    "One2One_F1AtSizeofGroundTruth",
    "f1_ground_size",
    "mean_f1_ground_size",
    "f1_at_ground_size",
    "f1_at_ground_size_mean",
    "recall_ground_size",
    "mean_recall_ground_size",
]

F1_1TO1_COLS = [
    "One2One_F1Score",
    "One2One_F1",
    "One2One_F1AtSizeofGroundTruth",
    "f1_1to1",
    "metric_f1_1to1",
    "mean_f1_1to1",
    "mean_f1_one2one",
    "mean_f1_one2one_pairs",
]

RUNTIME_COLS = [
    "runtime",
    "runtime_mean",
    "runtime_sec",
    "elapsed",
    "elapsed_time",
    "time",
]

# ============================================================
# HELPERS
# ============================================================

def normalize_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def pick_col(cols, candidates):
    exact = {str(c).lower(): c for c in cols}
    norm = {normalize_key(c): c for c in cols}

    for cand in candidates:
        if str(cand).lower() in exact:
            return exact[str(cand).lower()]

    for cand in candidates:
        key = normalize_key(cand)
        if key in norm:
            return norm[key]

    return None

def normalize_method(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return METHOD_MAP.get(x, x)

def method_from_path(path):
    p = str(path).lower()

    if "llmatch" in p:
        return "LLMATCH"
    if "smutf" in p:
        return "SMUTF"
    if "magnetoftgpt" in p or "magneto_ft_gpt" in p:
        return "MagnetoFTGPT"
    if "magnetogpt" in p or "magneto_no_ft_gpt" in p:
        return "MagnetoGPT"
    if "magnetoft" in p or "magneto_ft_no_gpt" in p:
        return "MagnetoFT"
    if "magneto" in p:
        return "Magneto"

    return np.nan

def make_pair_id(df):
    cols = set(df.columns)

    if "pair_id" in cols:
        return df["pair_id"].astype(str)

    if {"dataset", "source_table", "target_table"}.issubset(cols):
        return (
            df["dataset"].astype(str)
            + "__"
            + df["source_table"].astype(str)
            + "__"
            + df["target_table"].astype(str)
        )

    if {"benchmark", "dataset", "source_table", "target_table"}.issubset(cols):
        return (
            df["benchmark"].astype(str)
            + "__"
            + df["dataset"].astype(str)
            + "__"
            + df["source_table"].astype(str)
            + "__"
            + df["target_table"].astype(str)
        )

    if {"id_dataset", "source_table", "target_table"}.issubset(cols):
        return (
            df["id_dataset"].astype(str)
            + "__"
            + df["source_table"].astype(str)
            + "__"
            + df["target_table"].astype(str)
        )

    if {"dataset", "source", "target"}.issubset(cols):
        return (
            df["dataset"].astype(str)
            + "__"
            + df["source"].astype(str)
            + "__"
            + df["target"].astype(str)
        )

    if {"source", "target"}.issubset(cols):
        return df["source"].astype(str) + "__" + df["target"].astype(str)

    if "pair_name" in cols:
        return df["pair_name"].astype(str)

    return pd.Series([np.nan] * len(df), index=df.index)

def load_any_file(path):
    suffix = path.suffix.lower()

    try:
        if suffix == ".csv":
            return pd.read_csv(path, low_memory=False)

        if suffix == ".json":
            with open(path, "r") as f:
                data = json.load(f)
            if isinstance(data, dict):
                return pd.DataFrame([data])
            if isinstance(data, list):
                return pd.DataFrame(data)

        if suffix == ".jsonl":
            return pd.read_json(path, lines=True)

        if suffix == ".parquet":
            return pd.read_parquet(path)

        if suffix in [".xlsx", ".xls"]:
            return pd.read_excel(path)

    except Exception:
        return None

    return None

# ============================================================
# SCAN
# ============================================================

candidate_files = []
all_rows = []

for root in ROOTS:
    if not root.exists():
        print("Missing root:", root)
        continue

    for f in root.rglob("*"):
        if not f.is_file():
            continue

        if f.suffix.lower() not in [".csv", ".json", ".jsonl", ".parquet", ".xlsx", ".xls"]:
            continue

        df = load_any_file(f)
        if df is None or df.empty:
            continue

        cols = list(df.columns)

        method_col = pick_col(cols, ["method", "classifier", "method_norm", "matcher", "algorithm"])
        f1_col = pick_col(cols, F1_ALL_COLS)
        precision_col = pick_col(cols, PREC_ALL_COLS)
        recall_col = pick_col(cols, REC_ALL_COLS)
        gt_col = pick_col(cols, F1_GT_COLS)
        one_col = pick_col(cols, F1_1TO1_COLS)
        runtime_col = pick_col(cols, RUNTIME_COLS)

        if method_col is None and pd.isna(method_from_path(f)):
            continue

        if (
            f1_col is None
            and precision_col is None
            and recall_col is None
            and gt_col is None
            and one_col is None
        ):
            continue

        candidate_files.append(
            {
                "file": str(f),
                "method_col": method_col,
                "f1_col": f1_col,
                "precision_col": precision_col,
                "recall_col": recall_col,
                "ground_size_col": gt_col,
                "one_to_one_col": one_col,
                "runtime_col": runtime_col,
                "columns": " | ".join(cols),
            }
        )

        out = pd.DataFrame()

        if method_col is not None:
            out["method"] = df[method_col].map(normalize_method)
        else:
            out["method"] = method_from_path(f)

        out["pair_id"] = make_pair_id(df)
        out["source_file"] = str(f)

        out["f1_all_to_all"] = (
            pd.to_numeric(df[f1_col], errors="coerce") if f1_col else np.nan
        )
        out["precision_all_to_all"] = (
            pd.to_numeric(df[precision_col], errors="coerce") if precision_col else np.nan
        )
        out["recall_all_to_all"] = (
            pd.to_numeric(df[recall_col], errors="coerce") if recall_col else np.nan
        )
        out["f1_ground_size"] = (
            pd.to_numeric(df[gt_col], errors="coerce") if gt_col else np.nan
        )
        out["f1_one_to_one"] = (
            pd.to_numeric(df[one_col], errors="coerce") if one_col else np.nan
        )
        out["runtime"] = (
            pd.to_numeric(df[runtime_col], errors="coerce") if runtime_col else np.nan
        )

        out["used_f1_col"] = f1_col
        out["used_precision_col"] = precision_col
        out["used_recall_col"] = recall_col
        out["used_ground_size_col"] = gt_col
        out["used_one_to_one_col"] = one_col
        out["used_runtime_col"] = runtime_col

        out = out[
            ~out["method"]
            .astype(str)
            .str.lower()
            .str.contains("meta_space|meta space|metaspace|metamatch", na=False)
        ]

        out = out[out["method"].isin(KNOWN)]

        metric_cols = [
            "f1_all_to_all",
            "precision_all_to_all",
            "recall_all_to_all",
            "f1_ground_size",
            "f1_one_to_one",
        ]
        out = out[out[metric_cols].notna().any(axis=1)]

        if len(out):
            all_rows.append(out)

# ============================================================
# SAVE LONG FILES
# ============================================================

candidates = pd.DataFrame(candidate_files)
candidates.to_csv(OUT_CANDIDATES, index=False)

long = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
long.to_csv(OUT_LONG, index=False)

print("Candidate files:", len(candidates))
print("Loaded result rows:", len(long))

# ============================================================
# COVERAGE
# ============================================================

coverage = (
    long.groupby(["method", "source_file"])
    .agg(
        n_rows=("method", "size"),
        n_pairs=("pair_id", lambda x: x.dropna().nunique()),
        f1_all_mean=("f1_all_to_all", "mean"),
        f1_ground_mean=("f1_ground_size", "mean"),
        f1_one_to_one_mean=("f1_one_to_one", "mean"),
        precision_mean=("precision_all_to_all", "mean"),
        recall_mean=("recall_all_to_all", "mean"),
        runtime_mean=("runtime", "mean"),
        used_f1_col=("used_f1_col", lambda x: sorted(set(x.dropna().astype(str)))),
        used_ground_size_col=("used_ground_size_col", lambda x: sorted(set(x.dropna().astype(str)))),
        used_one_to_one_col=("used_one_to_one_col", lambda x: sorted(set(x.dropna().astype(str)))),
    )
    .reset_index()
)

coverage["dist_551"] = (coverage["n_pairs"] - 551).abs()

coverage = coverage.sort_values(
    ["method", "dist_551", "n_pairs", "f1_all_mean", "f1_ground_mean"],
    ascending=[True, True, False, False, False],
)

coverage.to_csv(OUT_COVERAGE, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================

final_rows = []

for method, g in long.groupby("method"):
    row = {"method": method}

    for metric, prefix in [
        ("f1_all_to_all", "all_to_all"),
        ("f1_ground_size", "ground_size"),
        ("f1_one_to_one", "one_to_one"),
    ]:
        gg = g[g[metric].notna()].copy()

        if gg.empty:
            row[f"f1_{prefix}_mean"] = np.nan
            row[f"f1_{prefix}_std"] = np.nan
            row[f"n_pairs_{prefix}"] = 0
            row[f"source_{prefix}"] = ""
            continue

        by_source = (
            gg.groupby("source_file")
            .agg(
                n_rows=("method", "size"),
                n_pairs=("pair_id", lambda x: x.dropna().nunique()),
                mean=(metric, "mean"),
                std=(metric, "std"),
            )
            .reset_index()
        )

        by_source["dist_551"] = (by_source["n_pairs"] - 551).abs()

        by_source = by_source.sort_values(
            ["dist_551", "n_pairs", "mean"],
            ascending=[True, False, False],
        )

        best_source = by_source.iloc[0]["source_file"]

        part = gg[gg["source_file"] == best_source].copy()

        if part["pair_id"].notna().any():
            part = (
                part.sort_values(metric, ascending=False)
                .drop_duplicates(subset=["method", "pair_id"], keep="first")
            )

        row[f"f1_{prefix}_mean"] = part[metric].mean()
        row[f"f1_{prefix}_std"] = part[metric].std()
        row[f"n_pairs_{prefix}"] = part["pair_id"].dropna().nunique()
        row[f"source_{prefix}"] = best_source

        if prefix == "all_to_all":
            row["precision_all_to_all_mean"] = part["precision_all_to_all"].mean()
            row["recall_all_to_all_mean"] = part["recall_all_to_all"].mean()
            row["runtime_mean"] = part["runtime"].mean()
            row["used_f1_col"] = (
                part["used_f1_col"].dropna().iloc[0]
                if part["used_f1_col"].notna().any()
                else None
            )

    final_rows.append(row)

summary = pd.DataFrame(final_rows)
summary = summary.sort_values("f1_all_to_all_mean", ascending=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("\nSaved:")
print(OUT_CANDIDATES)
print(OUT_LONG)
print(OUT_COVERAGE)
print(OUT_SUMMARY)

print("\nBest baseline summary:")
print(summary.to_string(index=False))

Candidate files: 247
Loaded result rows: 746732

Saved:
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/candidate_files.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/all_baseline_results_long.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/coverage_by_method_source.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_all_baselines_scan/all_baselines_best_summary.csv

Best baseline summary:
                 method  f1_all_to_all_mean  f1_all_to_all_std  n_pairs_all_to_all                                                                                                                                                                             source_all_to_all  precision_all_to_all_mean  recall_all_to_all_mean  runtime_mean     used_f1_col  f1_ground_size_mean  f1_ground_size_std  n_pairs_ground_size                                                                                                  

In [2]:
# ============================================================
# KEEP ONLY SI_SI SCENARIO WHEN AVAILABLE
# ============================================================

if "scenario" not in long.columns:
    # If scenario was not extracted, recover it from source filenames when possible
    long["scenario"] = np.nan

# Keep all rows without scenario, but when scenario exists, keep only si_si
long_sisi = long[
    long["scenario"].isna()
    | long["scenario"].astype(str).str.lower().str.strip().isin(["si_si", "sisi"])
].copy()

# ============================================================
# FINAL SUMMARY — BEST F1 PER METHOD / PAIR FOR SI_SI
# ============================================================

final_rows = []

for method, g in long_sisi.groupby("method"):
    row = {"method": method}

    for metric, prefix in [
        ("f1_all_to_all", "all_to_all"),
        ("f1_ground_size", "ground_size"),
        ("f1_one_to_one", "one_to_one"),
    ]:
        gg = g[g[metric].notna()].copy()

        if gg.empty:
            row[f"f1_{prefix}_mean"] = np.nan
            row[f"f1_{prefix}_std"] = np.nan
            row[f"n_pairs_{prefix}"] = 0
            row[f"source_{prefix}"] = ""
            continue

        # Remove obviously bad threshold files for baselines
        bad_mask = gg["source_file"].str.contains(
            "threshold_opt_train|pair_metrics_threshold_opt_train",
            case=False,
            na=False,
        )
        if (~bad_mask).any():
            gg = gg[~bad_mask].copy()

        # For each method/pair, keep the best F1 found anywhere
        if gg["pair_id"].notna().any():
            best_pair = (
                gg.sort_values(metric, ascending=False)
                  .drop_duplicates(subset=["method", "pair_id"], keep="first")
            )
        else:
            best_pair = gg.copy()

        row[f"f1_{prefix}_mean"] = best_pair[metric].mean()
        row[f"f1_{prefix}_std"] = best_pair[metric].std()
        row[f"n_pairs_{prefix}"] = best_pair["pair_id"].dropna().nunique()

        # keep source list
        row[f"source_{prefix}"] = " | ".join(
            sorted(best_pair["source_file"].dropna().astype(str).unique())[:10]
        )

        if prefix == "all_to_all":
            row["precision_all_to_all_mean"] = best_pair["precision_all_to_all"].mean()
            row["recall_all_to_all_mean"] = best_pair["recall_all_to_all"].mean()
            row["runtime_mean"] = best_pair["runtime"].mean()
            row["used_f1_cols"] = " | ".join(
                sorted(best_pair["used_f1_col"].dropna().astype(str).unique())
            )

    final_rows.append(row)

summary = pd.DataFrame(final_rows)
summary = summary.sort_values("f1_all_to_all_mean", ascending=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("\nBest SI_SI baseline summary:")
print(summary.to_string(index=False))


Best SI_SI baseline summary:
                 method  f1_all_to_all_mean  f1_all_to_all_std  n_pairs_all_to_all                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOTS = [
    Path("/Users/nahawandkired/Documents/Recherche/baselines"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs"),
    Path("/Users/nahawandkired/Downloads"),
]

METHOD_KEYS = [
    "Magneto",
    "MagnetoFT",
    "MagnetoGPT",
    "MagnetoFTGPT",
    "Magneto_no_ft_no_gpt",
    "Magneto_ft_no_gpt",
    "Magneto_no_ft_gpt",
    "Magneto_ft_gpt",
]

rows = []

def norm_method(x):
    x = str(x)
    return {
        "Magneto_no_ft_no_gpt": "Magneto",
        "Magneto_ft_no_gpt": "MagnetoFT",
        "Magneto_no_ft_gpt": "MagnetoGPT",
        "Magneto_ft_gpt": "MagnetoFTGPT",
    }.get(x, x)

for root in ROOTS:
    if not root.exists():
        continue

    for f in root.rglob("*"):
        if not f.is_file():
            continue
        if f.suffix.lower() not in [".csv", ".json", ".jsonl", ".parquet", ".xlsx"]:
            continue

        try:
            if f.suffix.lower() == ".csv":
                df = pd.read_csv(f, low_memory=False)
            elif f.suffix.lower() == ".jsonl":
                df = pd.read_json(f, lines=True)
            elif f.suffix.lower() == ".parquet":
                df = pd.read_parquet(f)
            elif f.suffix.lower() == ".xlsx":
                df = pd.read_excel(f)
            else:
                continue
        except Exception:
            continue

        text = " ".join(map(str, df.columns)) + " " + str(f)

        if "magneto" not in text.lower():
            continue

        method_col = None
        for c in ["method", "classifier", "method_norm"]:
            if c in df.columns:
                method_col = c
                break

        if method_col is not None:
            d = df[df[method_col].astype(str).str.contains("Magneto|magneto", case=False, na=False)].copy()
        else:
            d = df.copy()
            d["method"] = "Magneto_from_path"
            method_col = "method"

        if d.empty:
            continue

        # scenario complet-complet si présent
        if "scenario" in d.columns:
            d = d[d["scenario"].astype(str).str.lower().str.strip().eq("complet-complet")].copy()
            if d.empty:
                continue

        # pair id
        if "pair_name" in d.columns:
            pair = d["pair_name"].astype(str)
        elif "pair_id" in d.columns:
            pair = d["pair_id"].astype(str)
        elif {"dataset", "source_table", "target_table"}.issubset(d.columns):
            pair = d["dataset"].astype(str)+"__"+d["source_table"].astype(str)+"__"+d["target_table"].astype(str)
        else:
            pair = pd.Series([np.nan] * len(d))

        # f1 columns
        f1_cols = [c for c in d.columns if "f1" in c.lower()]
        useful = []
        for c in f1_cols:
            vals = pd.to_numeric(d[c], errors="coerce")
            if vals.notna().any():
                useful.append((c, vals.mean()))

        for m, g in d.groupby(method_col):
            mn = norm_method(m)

            for c, mean_val in useful:
                rows.append({
                    "method": mn,
                    "file": str(f),
                    "rows": len(g),
                    "n_pairs": pair.loc[g.index].dropna().nunique(),
                    "n_datasets": g["dataset"].nunique() if "dataset" in g.columns else np.nan,
                    "scenario_values": ",".join(sorted(g["scenario"].dropna().astype(str).unique())) if "scenario" in g.columns else "",
                    "f1_col": c,
                    "f1_mean": pd.to_numeric(g[c], errors="coerce").mean(),
                    "columns": " | ".join(d.columns),
                })

res = pd.DataFrame(rows)

out = "/Users/nahawandkired/Documents/metamatch/outputs/magneto_deep_search_all_dirs.csv"
res.to_csv(out, index=False)

print("Saved:", out)
print("Rows:", len(res))

if len(res):
    print(
        res.sort_values(["method", "n_pairs", "f1_mean"], ascending=[True, False, False])
           [["method","n_pairs","n_datasets","f1_col","f1_mean","file"]]
           .to_string(index=False)
    )


Saved: /Users/nahawandkired/Documents/metamatch/outputs/magneto_deep_search_all_dirs.csv
Rows: 499
           method  n_pairs  n_datasets                                  f1_col   f1_mean                                                                                                                                                                                                                                                        file
          Magneto      197         3.0                         One2One_F1Score  0.582961                                                                                                                                                         /Users/nahawandkired/Documents/Recherche/baselines/results/magneto_valentine_6scenarios.cleaned.csv
          Magneto      197         3.0                             All_F1Score  0.578210                                                                                                                                   

In [4]:

from pathlib import Path
import pandas as pd

ROOT = Path("/Users/nahawandkired")

for f in ROOT.rglob("*header_values*.csv"):
    try:
        df = pd.read_csv(f, low_memory=False)
    except:
        continue

    print("\n" + "="*80)
    print(f)

    for c in ["method","model","classifier"]:
        if c in df.columns:
            print("\nCOLUMN:", c)
            print(df[c].dropna().astype(str).value_counts().head(30))



/Users/nahawandkired/Documents/Recherche/magneto_matcher/results/benchmarks/valentine copy 2/OpenData/OpenData-opendata-header_values_repeat-exact_semantic_results_1.csv

COLUMN: method
method
Magneto         102
MagnetoGPT      101
MagnetoFT       101
MagnetoFTGPT    101
Name: count, dtype: int64

/Users/nahawandkired/Documents/Recherche/magneto_matcher/results/benchmarks/valentine copy 2/OpenData/OpenData-opendata-header_values_repeat-semantic_results_1.csv

COLUMN: method
method
Magneto         8
MagnetoGPT      8
MagnetoFT       7
MagnetoFTGPT    7
Name: count, dtype: int64

/Users/nahawandkired/Documents/Recherche/magneto_matcher/results/benchmarks/valentine copy 2/TPC-DI/TPC-DI-tpc-header_values_repeat-semantic_results_1.csv

COLUMN: method
method
Magneto         22
MagnetoGPT      22
MagnetoFT       22
MagnetoFTGPT    21
Name: count, dtype: int64

/Users/nahawandkired/Documents/Recherche/magneto_matcher/results/benchmarks/valentine copy 2/TPC-DI/TPC-DI-tpc-header_values_repeat-

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import os

FILES = [
    # Les 3 gros datasets complets
    Path("/Users/nahawandkired/Downloads/New project/resultats_presentation/resultats/magneto/ChEMBL-chembl-header_values_repeat-semantic_results_1.csv"),
    Path("/Users/nahawandkired/Downloads/New project/resultats_presentation/resultats/magneto/OpenData-opendata-header_values_repeat-exact_semantic_results_1.csv"),
    Path("/Users/nahawandkired/Downloads/New project/resultats_presentation/resultats/magneto/TPC-DI-tpc-header_values_repeat-exact_semantic_results_1.csv"),

    # Les petits datasets trouvés précédemment
    Path("/Users/nahawandkired/Downloads/New project/resultats_presentation/resultats/magneto/Magellan_results20260221095616.csv"),
    Path("/Users/nahawandkired/Downloads/New project/resultats_presentation/resultats/magneto/Wikidata_results20260221095612.csv"),
]

OUT = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_magneto_complet_complet_551.csv")
OUT_SUMMARY = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_magneto_complet_complet_551_summary.csv")

rows = []

for f in FILES:
    if not f.exists():
        print("MISSING:", f)
        continue

    df = pd.read_csv(f, low_memory=False)

    # garder seulement Magneto variants
    df = df[df["method"].astype(str).str.contains("Magneto", case=False, na=False)].copy()

    # pair id
    df["pair_id"] = (
        df["dataset"].astype(str) + "__" +
        df["source_table"].astype(str) + "__" +
        df["target_table"].astype(str)
    )

    # numériques
    for c in [
        "All_F1Score",
        "All_Precision",
        "All_Recall",
        "All_RecallAtSizeofGroundTruth",
        "One2One_F1Score",
        "runtime",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["source_file"] = str(f)
    rows.append(df)

all_df = pd.concat(rows, ignore_index=True)

# Déduplication : même méthode + même paire => garder le meilleur F1 all-to-all
all_df = (
    all_df.sort_values("All_F1Score", ascending=False)
          .drop_duplicates(subset=["method", "pair_id"], keep="first")
)

all_df.to_csv(OUT, index=False)

summary = (
    all_df.groupby("method")
          .agg(
              n_pairs=("pair_id", "nunique"),
              n_datasets=("dataset", "nunique"),
              f1_all_to_all_mean=("All_F1Score", "mean"),
              f1_all_to_all_std=("All_F1Score", "std"),
              precision_mean=("All_Precision", "mean"),
              recall_mean=("All_Recall", "mean"),
              f1_gt_mean=("All_RecallAtSizeofGroundTruth", "mean"),
              f1_gt_std=("All_RecallAtSizeofGroundTruth", "std"),
              f1_1to1_mean=("One2One_F1Score", "mean"),
              f1_1to1_std=("One2One_F1Score", "std"),
              runtime_mean=("runtime", "mean"),
          )
          .reset_index()
          .sort_values("f1_all_to_all_mean", ascending=False)
)

summary.to_csv(OUT_SUMMARY, index=False)

print("Saved:", OUT)
print("Saved:", OUT_SUMMARY)
print()
print(summary.to_string(index=False))

print("\nCoverage by dataset/method:")
print(
    all_df.groupby(["dataset", "method"])["pair_id"]
          .nunique()
          .unstack(fill_value=0)
          .to_string()
)


Saved: /Users/nahawandkired/Documents/metamatch/outputs/final_magneto_complet_complet_551.csv
Saved: /Users/nahawandkired/Documents/metamatch/outputs/final_magneto_complet_complet_551_summary.csv

      method  n_pairs  n_datasets  f1_all_to_all_mean  f1_all_to_all_std  precision_mean  recall_mean  f1_gt_mean  f1_gt_std  f1_1to1_mean  f1_1to1_std  runtime_mean
MagnetoFTGPT      551           5            0.742552           0.247992        0.759627     0.845023    0.801404   0.236388      0.731086     0.232615     61.992296
  MagnetoGPT      551           5            0.724373           0.293005        0.696395     0.903064    0.741450   0.309185      0.727193     0.277868    228.689958
   MagnetoFT      551           5            0.611466           0.266639        0.559678     0.804297    0.755121   0.265607      0.621443     0.229256      9.299979
     Magneto      551           5            0.580057           0.288001        0.508483     0.840872    0.750642   0.282488      0.599415 

In [6]:

from pathlib import Path
import pandas as pd

ROOTS = [
    Path("/Users/nahawandkired/Documents/Recherche"),
    Path("/Users/nahawandkired/Documents/metamatch/outputs"),
    Path("/Users/nahawandkired/Downloads"),
]

KEYWORDS = [
    "cupid",
    "coma",
    "distribution",
    "flood",
    "isresmat",
]

for root in ROOTS:
    for f in root.rglob("*.csv"):
        p = str(f).lower()

        if not any(k in p for k in KEYWORDS):
            continue

        try:
            df = pd.read_csv(f, nrows=2)
        except:
            continue

        print("\n" + "="*120)
        print(f)
        print(df.columns.tolist())



/Users/nahawandkired/Documents/Recherche/baselines/results/llmatch_cupid_valentine.csv
['dataset', 'category', 'pair_name', 'method', 'runtime', 'mrr', 'All_Precision', 'All_F1Score', 'All_Recall', 'All_PrecisionTop10Percent', 'All_RecallAtSizeofGroundTruth', 'One2One_Precision', 'One2One_F1Score', 'One2One_Recall', 'One2One_PrecisionTop10Percent', 'One2One_RecallAtSizeofGroundTruth', 'error']

/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_repeats_zero_baselines_fixed_01/run_01/results/fold_0/coma_pp/test_manifest.csv
['pair_id', 'source_col_norm', 'target_col_norm', 'label']

/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_repeats_zero_baselines_fixed_01/run_01/results/fold_0/coma_inst/test_manifest.csv
['pair_id', 'source_col_norm', 'target_col_norm', 'label']

/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata_repeats_zero_baselines_fixed_01/run_01/results/fold_0/cupid_ext/test_manifest.csv
['pair_id', 'source_col_norm', 'target_col_norm', '

In [7]:

import pandas as pd

f = "/Users/nahawandkired/Documents/Recherche/baselines/results/llmatch_cupid_valentine.csv"

df = pd.read_csv(f)

print(df["method"].value_counts())

print()
print("datasets")
print(df["dataset"].value_counts())

print()
print("methods")
print(sorted(df["method"].unique()))


method
LLMATCH_CUPID    15
Name: count, dtype: int64

datasets
dataset
ChEMBL    15
Name: count, dtype: int64

methods
['LLMATCH_CUPID']


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []

def add_common(df, method_col, dataset_col, pair_col, f1_col, gt_col, one_col, precision_col=None, recall_col=None, runtime_col=None, source="", scenario=""):
    d = df.copy()
    out = pd.DataFrame()
    out["method"] = d[method_col].astype(str)
    out["dataset"] = d[dataset_col].astype(str)
    out["pair_id"] = d[pair_col].astype(str)
    out["f1_all_to_all"] = pd.to_numeric(d[f1_col], errors="coerce")
    out["f1_at_ground_size"] = pd.to_numeric(d[gt_col], errors="coerce") if gt_col in d.columns else np.nan
    out["f1_one_to_one"] = pd.to_numeric(d[one_col], errors="coerce") if one_col in d.columns else np.nan
    out["precision"] = pd.to_numeric(d[precision_col], errors="coerce") if precision_col and precision_col in d.columns else np.nan
    out["recall"] = pd.to_numeric(d[recall_col], errors="coerce") if recall_col and recall_col in d.columns else np.nan
    out["runtime"] = pd.to_numeric(d[runtime_col], errors="coerce") if runtime_col and runtime_col in d.columns else np.nan
    out["scenario"] = scenario
    out["source_file"] = source
    return out

# 1) LLMATCH complet-complet
f = "/Users/nahawandkired/Documents/Recherche/baselines/results/llmatch_valentine_6_scenarios.csv"
df = pd.read_csv(f, low_memory=False, on_bad_lines="skip")
df = df[df["scenario"].astype(str).str.lower().str.strip().eq("complet-complet")].copy()
rows.append(add_common(df, "method", "dataset", "pair_name", "All_F1Score", "All_F1AtSizeofGroundTruth", "One2One_F1Score", "All_Precision", "All_Recall", "runtime", f, "complet-complet"))

# 2) SMUTF complet-complet
f = "/Users/nahawandkired/Documents/Recherche/baselines/results/smutf_valentine_6_scenarios.csv"
df = pd.read_csv(f, low_memory=False, on_bad_lines="skip")
df = df[df["scenario"].astype(str).str.lower().str.strip().eq("complet-complet")].copy()
rows.append(add_common(df, "method", "dataset", "pair_name", "All_F1Score", "All_F1AtSizeofGroundTruth", "One2One_F1Score", "All_Precision", "All_Recall", "runtime", f, "complet-complet"))

# 3) Magneto reconstruit 551
f = "/Users/nahawandkired/Documents/metamatch/outputs/final_magneto_complet_complet_551.csv"
df = pd.read_csv(f, low_memory=False, on_bad_lines="skip")
df["pair_id"] = df["dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)
rows.append(add_common(df, "method", "dataset", "pair_id", "All_F1Score", "All_RecallAtSizeofGroundTruth", "One2One_F1Score", "All_Precision", "All_Recall", "runtime", f, "complet-complet/header_values"))

# 4) Baselines classiques si_si
f = "/Users/nahawandkired/Documents/Recherche/baselines/data_local/baseline/baseline.csv"
df = pd.read_csv(f, low_memory=False, on_bad_lines="skip")
df = df[df["scenario"].astype(str).str.lower().str.strip().eq("si_si")].copy()

# on garde seulement les classiques ici, car Magneto est déjà mieux reconstruit au-dessus
classic = ["Coma", "Coma++", "ComaInst", "Cupid", "Distribution", "ISResMat", "SimilarityFlooding"]
df = df[df["method"].isin(classic)].copy()
df["pair_id"] = df["dataset"].astype(str) + "__" + df["source_table"].astype(str) + "__" + df["target_table"].astype(str)

rows.append(add_common(df, "method", "dataset", "pair_id", "All_F1Score", "All_RecallAtSizeofGroundTruth", "One2One_F1Score", "All_Precision", "All_Recall", "runtime", f, "si_si"))

# UNION
all_results = pd.concat(rows, ignore_index=True)

# normaliser noms
all_results["method"] = all_results["method"].replace({
    "SimilarityFlooding": "Similarity Flooding",
    "Coma": "COMA",
    "Coma++": "COMA++",
    "ComaInst": "COMA Instance",
})

# dédoublonner : même méthode + même paire => garder meilleur F1
all_results = (
    all_results.sort_values("f1_all_to_all", ascending=False)
               .drop_duplicates(subset=["method", "pair_id"], keep="first")
)

# summary dataset
by_dataset = (
    all_results.groupby(["method", "dataset", "scenario"])
    .agg(
        n_pairs=("pair_id", "nunique"),
        f1_all_to_all_mean=("f1_all_to_all", "mean"),
        f1_all_to_all_std=("f1_all_to_all", "std"),
        f1_at_ground_size_mean=("f1_at_ground_size", "mean"),
        f1_at_ground_size_std=("f1_at_ground_size", "std"),
        f1_one_to_one_mean=("f1_one_to_one", "mean"),
        f1_one_to_one_std=("f1_one_to_one", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        runtime_mean=("runtime", "mean"),
    )
    .reset_index()
    .sort_values(["f1_all_to_all_mean"], ascending=False)
)

# summary méthode
by_method = (
    all_results.groupby(["method", "scenario"])
    .agg(
        n_pairs=("pair_id", "nunique"),
        n_datasets=("dataset", "nunique"),
        f1_all_to_all_mean=("f1_all_to_all", "mean"),
        f1_all_to_all_std=("f1_all_to_all", "std"),
        f1_at_ground_size_mean=("f1_at_ground_size", "mean"),
        f1_at_ground_size_std=("f1_at_ground_size", "std"),
        f1_one_to_one_mean=("f1_one_to_one", "mean"),
        f1_one_to_one_std=("f1_one_to_one", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        runtime_mean=("runtime", "mean"),
    )
    .reset_index()
    .sort_values("f1_all_to_all_mean", ascending=False)
)

all_results.to_csv(OUT_DIR / "final_all_results_pair_level.csv", index=False)
by_dataset.to_csv(OUT_DIR / "final_all_results_by_dataset.csv", index=False)
by_method.to_csv(OUT_DIR / "final_all_results_by_method.csv", index=False)

print("Saved:")
print(OUT_DIR / "final_all_results_pair_level.csv")
print(OUT_DIR / "final_all_results_by_dataset.csv")
print(OUT_DIR / "final_all_results_by_method.csv")

print("\nBy method:")
print(by_method.to_string(index=False))


Saved:
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/final_all_results_pair_level.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/final_all_results_by_dataset.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/final_all_results_by_method.csv

By method:
             method                      scenario  n_pairs  n_datasets  f1_all_to_all_mean  f1_all_to_all_std  f1_at_ground_size_mean  f1_at_ground_size_std  f1_one_to_one_mean  f1_one_to_one_std  precision_mean  recall_mean  runtime_mean
        LLMatch-llm               complet-complet      551           5            0.853242           0.245641                0.944036               0.168749            0.864735           0.237188        0.837152     0.933294     24.150912
       MagnetoFTGPT complet-complet/header_values      551           5            0.742552           0.247992                0.801404               0.236388

In [11]:

from pathlib import Path
import pandas as pd
import json
import numpy as np
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score

ROOT = Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results")
OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def norm(x):
    return str(x).strip().lower()

rows_pair = []
rows_fold = []

for fold_dir in sorted(ROOT.glob("fold_*/ISResMat")):
    fold = fold_dir.parent.name

    pred_f = fold_dir / "predictions.csv"
    manifest_f = fold_dir / "pair_manifest.jsonl"

    if not pred_f.exists() or not manifest_f.exists():
        continue

    preds = pd.read_csv(pred_f)
    preds["source_col_norm"] = preds["source_col_norm"].map(norm)
    preds["target_col_norm"] = preds["target_col_norm"].map(norm)

    manifest = []
    with open(manifest_f) as fh:
        for line in fh:
            manifest.append(json.loads(line))

    pair_to_gt = {}

    for item in manifest:
        pair_id = item["pair_id"]
        mapping_json = item["mapping_json"]

        try:
            data = json.load(open(mapping_json))
        except Exception:
            pair_to_gt[pair_id] = set()
            continue

        gt = set()
        for m in data.get("matches", []):
            gt.add((norm(m["source_column"]), norm(m["target_column"])))

        pair_to_gt[pair_id] = gt

    preds["is_true"] = preds.apply(
        lambda r: (r["source_col_norm"], r["target_col_norm"]) in pair_to_gt.get(r["pair_id"], set()),
        axis=1
    ).astype(int)

    y_true = preds["is_true"].values
    scores = pd.to_numeric(preds["score"], errors="coerce").fillna(-np.inf).values

    # seuil optimal global sur le fold
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)

    best_idx = int(np.nanargmax(f1s))
    best_f1 = float(f1s[best_idx])
    best_precision = float(precision[best_idx])
    best_recall = float(recall[best_idx])

    if best_idx < len(thresholds):
        best_threshold = float(thresholds[best_idx])
    else:
        best_threshold = float(np.max(scores))

    y_pred = (scores >= best_threshold).astype(int)

    # métriques par pair_id au seuil optimal du fold
    for pair_id, g in preds.assign(y_pred=y_pred).groupby("pair_id"):
        yt = g["is_true"].values
        yp = g["y_pred"].values
        sc = g["score"].values

        if yt.sum() == 0:
            continue

        rows_pair.append({
            "method": "ISResMat",
            "scenario": "si_si",
            "fold": fold,
            "pair_id": pair_id,
            "dataset": pair_id.split("__")[0],
            "n_candidates": len(g),
            "n_true": int(yt.sum()),
            "threshold": best_threshold,
            "f1_all_to_all": f1_score(yt, yp, zero_division=0),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "source_file": str(pred_f),
        })

    rows_fold.append({
        "method": "ISResMat",
        "scenario": "si_si",
        "fold": fold,
        "n_rows": len(preds),
        "n_pairs": preds["pair_id"].nunique(),
        "n_positive": int(y_true.sum()),
        "best_threshold": best_threshold,
        "f1_all_to_all_opt_threshold": best_f1,
        "precision_opt_threshold": best_precision,
        "recall_opt_threshold": best_recall,
        "source_file": str(pred_f),
    })

pair_df = pd.DataFrame(rows_pair)
fold_df = pd.DataFrame(rows_fold)

pair_out = OUT_DIR / "isresmat_opt_threshold_pair_level.csv"
fold_out = OUT_DIR / "isresmat_opt_threshold_fold_level.csv"
summary_out = OUT_DIR / "isresmat_opt_threshold_summary.csv"

pair_df.to_csv(pair_out, index=False)
fold_df.to_csv(fold_out, index=False)

summary = pd.DataFrame([{
    "method": "ISResMat",
    "scenario": "si_si",
    "n_pairs": pair_df["pair_id"].nunique(),
    "n_datasets": pair_df["dataset"].nunique(),
    "f1_all_to_all_mean_pair": pair_df["f1_all_to_all"].mean(),
    "f1_all_to_all_std_pair": pair_df["f1_all_to_all"].std(),
    "precision_mean_pair": pair_df["precision"].mean(),
    "recall_mean_pair": pair_df["recall"].mean(),
    "f1_all_to_all_mean_fold": fold_df["f1_all_to_all_opt_threshold"].mean(),
    "f1_all_to_all_std_fold": fold_df["f1_all_to_all_opt_threshold"].std(),
    "precision_mean_fold": fold_df["precision_opt_threshold"].mean(),
    "recall_mean_fold": fold_df["recall_opt_threshold"].mean(),
    "threshold_mean": fold_df["best_threshold"].mean(),
    "threshold_std": fold_df["best_threshold"].std(),
    "source": str(ROOT / "fold_*/ISResMat/predictions.csv"),
    "metric_source": "optimal_threshold_from_predictions_and_mapping_json",
}])

summary.to_csv(summary_out, index=False)

print("Saved:")
print(pair_out)
print(fold_out)
print(summary_out)

print("\nFold-level:")
print(fold_df.to_string(index=False))

print("\nSummary:")
print(summary.to_string(index=False))


/Users/nahawandkired/Documents/metamatch/.venv/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/nahawandkired/Documents/metamatch/.venv/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Saved:
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/isresmat_opt_threshold_pair_level.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/isresmat_opt_threshold_fold_level.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/isresmat_opt_threshold_summary.csv

Fold-level:
  method scenario   fold  n_rows  n_pairs  n_positive  best_threshold  f1_all_to_all_opt_threshold  precision_opt_threshold  recall_opt_threshold                                                                                           source_file
ISResMat    si_si fold_0  118124      166        2693        0.538260                     0.332681                 0.248039              0.505013 /Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/ISResMat/predictions.csv
ISResMat    si_si fold_1  118852      166        2702        0.538260                     0.344951                 0.263818 

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score

# ============================================================
# CONFIG
# ============================================================

ROOT = Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results")
OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# NORMALISATION FORTE
# lowercase + suppression espaces, underscore, ponctuation
# ============================================================

def norm(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower().strip())

# ============================================================
# RECONSTRUCTION ISResMat
# predictions.csv + pair_manifest.jsonl + mapping_json
# ============================================================

rows_pair = []
rows_fold = []

for fold_dir in sorted(ROOT.glob("fold_*/ISResMat")):
    fold = fold_dir.parent.name

    pred_f = fold_dir / "predictions.csv"
    manifest_f = fold_dir / "pair_manifest.jsonl"

    if not pred_f.exists() or not manifest_f.exists():
        print("Missing:", fold_dir)
        continue

    print("Processing:", fold)

    preds = pd.read_csv(pred_f)

    required_cols = {"pair_id", "source_col_norm", "target_col_norm", "score"}
    missing = required_cols - set(preds.columns)
    if missing:
        print("Missing columns:", missing, "in", pred_f)
        continue

    preds["source_col_key"] = preds["source_col_norm"].map(norm)
    preds["target_col_key"] = preds["target_col_norm"].map(norm)
    preds["score"] = pd.to_numeric(preds["score"], errors="coerce").fillna(-np.inf)

    # charger manifest : pair_id -> mapping_json
    manifest = []
    with open(manifest_f) as fh:
        for line in fh:
            manifest.append(json.loads(line))

    pair_to_gt = {}

    for item in manifest:
        pair_id = item["pair_id"]
        mapping_json = item.get("mapping_json")

        gt = set()

        if mapping_json and Path(mapping_json).exists():
            try:
                data = json.load(open(mapping_json))
                for m in data.get("matches", []):
                    gt.add((norm(m.get("source_column")), norm(m.get("target_column"))))
            except Exception as e:
                print("Mapping error:", mapping_json, e)

        pair_to_gt[pair_id] = gt

    preds["is_true"] = preds.apply(
        lambda r: (
            r["source_col_key"],
            r["target_col_key"]
        ) in pair_to_gt.get(r["pair_id"], set()),
        axis=1
    ).astype(int)

    y_true = preds["is_true"].values
    scores = preds["score"].values

    n_positive = int(y_true.sum())

    # sécurité : si aucun positif, on ne cherche pas de seuil
    if n_positive == 0:
        print("WARNING: no positive labels in", fold)
        rows_fold.append({
            "method": "ISResMat",
            "scenario": "si_si",
            "fold": fold,
            "n_rows": len(preds),
            "n_pairs": preds["pair_id"].nunique(),
            "n_positive": 0,
            "best_threshold": np.nan,
            "f1_all_to_all_opt_threshold": np.nan,
            "precision_opt_threshold": np.nan,
            "recall_opt_threshold": np.nan,
            "source_file": str(pred_f),
        })
        continue

    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)

    best_idx = int(np.nanargmax(f1s))
    best_f1 = float(f1s[best_idx])
    best_precision = float(precision[best_idx])
    best_recall = float(recall[best_idx])

    if best_idx < len(thresholds):
        best_threshold = float(thresholds[best_idx])
    else:
        best_threshold = float(np.max(scores))

    preds["y_pred"] = (preds["score"] >= best_threshold).astype(int)

    # métriques par paire
    for pair_id, g in preds.groupby("pair_id"):
        yt = g["is_true"].values
        yp = g["y_pred"].values

        if yt.sum() == 0:
            continue

        rows_pair.append({
            "method": "ISResMat",
            "scenario": "si_si",
            "fold": fold,
            "pair_id": pair_id,
            "dataset": pair_id.split("__")[0],
            "n_candidates": len(g),
            "n_true": int(yt.sum()),
            "threshold": best_threshold,
            "f1_all_to_all": f1_score(yt, yp, zero_division=0),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "source_file": str(pred_f),
        })

    rows_fold.append({
        "method": "ISResMat",
        "scenario": "si_si",
        "fold": fold,
        "n_rows": len(preds),
        "n_pairs": preds["pair_id"].nunique(),
        "n_positive": n_positive,
        "best_threshold": best_threshold,
        "f1_all_to_all_opt_threshold": best_f1,
        "precision_opt_threshold": best_precision,
        "recall_opt_threshold": best_recall,
        "source_file": str(pred_f),
    })

# ============================================================
# SAVE RESULTS
# ============================================================

pair_df = pd.DataFrame(rows_pair)
fold_df = pd.DataFrame(rows_fold)

pair_out = OUT_DIR / "isresmat_opt_threshold_pair_level_STRONG_NORM.csv"
fold_out = OUT_DIR / "isresmat_opt_threshold_fold_level_STRONG_NORM.csv"
summary_out = OUT_DIR / "isresmat_opt_threshold_summary_STRONG_NORM.csv"

pair_df.to_csv(pair_out, index=False)
fold_df.to_csv(fold_out, index=False)

summary = pd.DataFrame([{
    "method": "ISResMat",
    "scenario": "si_si",
    "n_pairs": pair_df["pair_id"].nunique() if len(pair_df) else 0,
    "n_datasets": pair_df["dataset"].nunique() if len(pair_df) else 0,
    "f1_all_to_all_mean_pair": pair_df["f1_all_to_all"].mean() if len(pair_df) else np.nan,
    "f1_all_to_all_std_pair": pair_df["f1_all_to_all"].std() if len(pair_df) else np.nan,
    "precision_mean_pair": pair_df["precision"].mean() if len(pair_df) else np.nan,
    "recall_mean_pair": pair_df["recall"].mean() if len(pair_df) else np.nan,
    "f1_all_to_all_mean_fold": fold_df["f1_all_to_all_opt_threshold"].mean(),
    "f1_all_to_all_std_fold": fold_df["f1_all_to_all_opt_threshold"].std(),
    "precision_mean_fold": fold_df["precision_opt_threshold"].mean(),
    "recall_mean_fold": fold_df["recall_opt_threshold"].mean(),
    "threshold_mean": fold_df["best_threshold"].mean(),
    "threshold_std": fold_df["best_threshold"].std(),
    "source": str(ROOT / "fold_*/ISResMat/predictions.csv"),
    "metric_source": "optimal_threshold_from_predictions_and_mapping_json_strong_norm",
}])

summary.to_csv(summary_out, index=False)

print("\nSaved:")
print(pair_out)
print(fold_out)
print(summary_out)

print("\nFold-level:")
print(fold_df.to_string(index=False))

print("\nSummary:")
print(summary.to_string(index=False))

print("\nCoverage by dataset:")
if len(pair_df):
    print(pair_df.groupby("dataset")["pair_id"].nunique().to_string())
else:
    print("No pair-level results.")

Processing: fold_0
Processing: fold_1
Processing: fold_2
Processing: fold_3
Processing: fold_4
Processing: fold_5

Saved:
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/isresmat_opt_threshold_pair_level_STRONG_NORM.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/isresmat_opt_threshold_fold_level_STRONG_NORM.csv
/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL/isresmat_opt_threshold_summary_STRONG_NORM.csv

Fold-level:
  method scenario   fold  n_rows  n_pairs  n_positive  best_threshold  f1_all_to_all_opt_threshold  precision_opt_threshold  recall_opt_threshold                                                                                           source_file
ISResMat    si_si fold_0  118124      166        2693        0.538260                     0.332681                 0.248039              0.505013 /Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_

In [13]:
import pandas as pd

f = "/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/cupid/predictions.parquet"

df = pd.read_parquet(f)

print(df.head())
print()
print(df.columns.tolist())
print()
print(df.shape)

                                     pair_id dataset relation_type category  \
0  ChEMBL__Joinable__assays_both_50_1_ac2_ev  ChEMBL      Joinable   ChEMBL   
1  ChEMBL__Joinable__assays_both_50_1_ac2_ev  ChEMBL      Joinable   ChEMBL   
2  ChEMBL__Joinable__assays_both_50_1_ac2_ev  ChEMBL      Joinable   ChEMBL   
3  ChEMBL__Joinable__assays_both_50_1_ac2_ev  ChEMBL      Joinable   ChEMBL   
4  ChEMBL__Joinable__assays_both_50_1_ac2_ev  ChEMBL      Joinable   ChEMBL   

              source_table             target_table source_column  \
0  assays_both_50_1_ac2_ev  assays_both_50_1_ac2_ev      assay_id   
1  assays_both_50_1_ac2_ev  assays_both_50_1_ac2_ev      assay_id   
2  assays_both_50_1_ac2_ev  assays_both_50_1_ac2_ev      assay_id   
3  assays_both_50_1_ac2_ev  assays_both_50_1_ac2_ev      assay_id   
4  assays_both_50_1_ac2_ev  assays_both_50_1_ac2_ev      assay_id   

  target_column source_col_norm target_col_norm  ...  tda_h1_entropy_combined  \
0           ASI        assay_

In [14]:
for method in [
    "cupid",
    "similarity_flooding",
    "distribution_based",
    "coma",
    "coma_pp",
]:
    f = f"/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/{method}/predictions.parquet"

    try:
        df = pd.read_parquet(f)

        print("\n", "="*80)
        print(method)

        print("rows:", len(df))
        print("score nunique:", df["score"].nunique())
        print(df["score"].describe())

    except Exception as e:
        print(method, e)


cupid
rows: 118124
score nunique: 1
count    118124.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: score, dtype: float64

similarity_flooding
rows: 118124
score nunique: 26302
count    118124.000000
mean          0.038026
std           0.020048
min           0.016333
25%           0.024108
50%           0.029702
75%           0.047186
max           0.386514
Name: score, dtype: float64

distribution_based
rows: 118124
score nunique: 6272
count    118124.000000
mean          0.136793
std           0.333169
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: score, dtype: float64

coma
rows: 118124
score nunique: 1
count    118124.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: score, dtype: float64

coma_pp
rows: 118124
score nunique: 1
count    118

In [15]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/nahawandkired/Documents/metamatch/outputs")

METHODS = [
    "coma",
    "coma_pp",
    "similarity_flooding",
    "distribution_based",
    "isresmat",
]

for m in METHODS:

    print("\n" + "="*100)
    print(m)

    files = []

    for f in ROOT.rglob("*"):

        name = str(f).lower()

        if m in name and (
            "metrics" in name
            or "summary" in name
            or "results" in name
            or name.endswith(".csv")
            or name.endswith(".json")
        ):
            files.append(f)

    print("found:", len(files))

    for f in sorted(files)[:50]:
        print(f)


coma
found: 1282
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma/metrics.json
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma/pair_manifest.jsonl
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma/predictions.parquet
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma/task_info.json
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma/test_manifest.csv
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma_inst
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma_inst/metrics.json
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma_inst/pair_manifest.jsonl
/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results/fold_0/coma_inst/predictions.parquet
/Us

In [16]:
from pathlib import Path
import pandas as pd

ROOT = Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results")

for method in [
    "coma",
    "coma_pp",
    "similarity_flooding",
    "distribution_based",
    "isresmat",
]:
    f = ROOT / "fold_0" / method / "predictions.parquet"

    if not f.exists():
        continue

    df = pd.read_parquet(f)

    print("\n", method)
    print("pairs:", df["pair_id"].nunique())
    print("has label:", "label" in df.columns)
    print("has score:", "score" in df.columns)


 coma
pairs: 166
has label: True
has score: True

 coma_pp
pairs: 166
has label: True
has score: True

 similarity_flooding
pairs: 166
has label: True
has score: True

 distribution_based
pairs: 166
has label: True
has score: True

 isresmat
pairs: 166
has label: True
has score: True


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

ROOT = Path("/Users/nahawandkired/Documents/metamatch/outputs/exp_occidata/results")
OUT_DIR = Path("/Users/nahawandkired/Documents/metamatch/outputs/final_consolidated_baselines_FULL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_MAP = {
    "coma": "COMA",
    "coma_pp": "COMA++",
    "similarity_flooding": "Similarity Flooding",
    "distribution_based": "Distribution",
    "isresmat": "ISResMat",
}

rows_pair = []

def one_to_one_metrics(g):
    pred = (
        g.sort_values("score", ascending=False)
         .drop_duplicates("source_col_norm")
         .drop_duplicates("target_col_norm")
    )
    yt = pred["label"].astype(int).values
    yp = np.ones(len(pred), dtype=int)

    return {
        "One2One_Precision": precision_score(yt, yp, zero_division=0),
        "One2One_Recall": recall_score(g["label"].astype(int).values, g.index.isin(pred.index).astype(int), zero_division=0),
        "One2One_F1Score": f1_score(g["label"].astype(int).values, g.index.isin(pred.index).astype(int), zero_division=0),
    }

for method_dir, method_name in METHOD_MAP.items():
    print("Processing", method_name)

    for fold_dir in sorted(ROOT.glob("fold_*")):
        f = fold_dir / method_dir / "predictions.parquet"
        if not f.exists():
            continue

        df = pd.read_parquet(f)

        required = {"pair_id", "dataset", "source_table", "target_table", "source_col_norm", "target_col_norm", "label", "score"}
        missing = required - set(df.columns)
        if missing:
            print("Missing", missing, "in", f)
            continue

        df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
        df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0.0)

        # seuil optimal par fold
        best_t, best_f1 = 0.5, -1
        for t in np.unique(df["score"].values):
            yp = (df["score"].values >= t).astype(int)
            f1 = f1_score(df["label"].values, yp, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)

        df["pred"] = (df["score"] >= best_t).astype(int)

        for pair_id, g in df.groupby("pair_id"):
            y_true = g["label"].values
            y_pred = g["pred"].values

            if y_true.sum() == 0:
                continue

            top_gt = g.sort_values("score", ascending=False).head(int(y_true.sum()))
            recall_at_gt = top_gt["label"].sum() / y_true.sum()

            one = one_to_one_metrics(g)

            rows_pair.append({
                "benchmark": "Valentine",
                "scenario": "si_si",
                "dataset": g["dataset"].iloc[0],
                "source_table": g["source_table"].iloc[0],
                "target_table": g["target_table"].iloc[0],
                "method": method_name,
                "fold": fold_dir.name,
                "pair_id": pair_id,
                "n_candidates": len(g),
                "nmatches": int(y_true.sum()),
                "threshold": best_t,
                "All_Precision": precision_score(y_true, y_pred, zero_division=0),
                "All_F1Score": f1_score(y_true, y_pred, zero_division=0),
                "All_Recall": recall_score(y_true, y_pred, zero_division=0),
                "All_RecallAtSizeofGroundTruth": recall_at_gt,
                "One2One_Precision": one["One2One_Precision"],
                "One2One_F1Score": one["One2One_F1Score"],
                "One2One_Recall": one["One2One_Recall"],
                "source_file": str(f),
            })

pair_df = pd.DataFrame(rows_pair)

pair_out = OUT_DIR / "recomputed_missing_valentine_pair_level.csv"
dataset_out = OUT_DIR / "recomputed_missing_valentine_by_dataset.csv"
method_out = OUT_DIR / "recomputed_missing_valentine_by_method.csv"

pair_df.to_csv(pair_out, index=False)

by_dataset = (
    pair_df.groupby(["method", "dataset"])
    .agg(
        n_pairs=("pair_id", "nunique"),
        All_F1Score_mean=("All_F1Score", "mean"),
        All_F1Score_std=("All_F1Score", "std"),
        All_RecallAtSizeofGroundTruth_mean=("All_RecallAtSizeofGroundTruth", "mean"),
        One2One_F1Score_mean=("One2One_F1Score", "mean"),
        All_Precision_mean=("All_Precision", "mean"),
        All_Recall_mean=("All_Recall", "mean"),
    )
    .reset_index()
)

by_method = (
    pair_df.groupby("method")
    .agg(
        n_pairs=("pair_id", "nunique"),
        n_datasets=("dataset", "nunique"),
        All_F1Score_mean=("All_F1Score", "mean"),
        All_F1Score_std=("All_F1Score", "std"),
        All_RecallAtSizeofGroundTruth_mean=("All_RecallAtSizeofGroundTruth", "mean"),
        One2One_F1Score_mean=("One2One_F1Score", "mean"),
        All_Precision_mean=("All_Precision", "mean"),
        All_Recall_mean=("All_Recall", "mean"),
    )
    .reset_index()
    .sort_values("All_F1Score_mean", ascending=False)
)

by_dataset.to_csv(dataset_out, index=False)
by_method.to_csv(method_out, index=False)

print("Saved:")
print(pair_out)
print(dataset_out)
print(method_out)

print("\nBy method:")
print(by_method.to_string(index=False))

print("\nCoverage by dataset:")
print(pair_df.groupby(["method", "dataset"])["pair_id"].nunique().unstack(fill_value=0).to_string())

Processing COMA
Processing COMA++
Processing Similarity Flooding
Processing Distribution


KeyboardInterrupt: 